# Reproducing Peterson et al. (2021) from dandiset 000055

**Paper.** Peterson SM, Singh SH, Wang NXR, Rao RPN, Brunton BW (2021). Behavioral and neural
variability of naturalistic arm movements. *eNeuro* 8(3), ENEURO.0007-21.2021.
[doi:10.1523/ENEURO.0007-21.2021](https://doi.org/10.1523/ENEURO.0007-21.2021). The claims file was
written from the version of record in Europe PMC (PMC8225404): main text, Methods, legends, Tables 1
and 2 and the legends of the Extended Data figures.

**Dataset.** [DANDI:000055](https://dandiarchive.org/dandiset/000055/0.220127.0436) (AJILE12),
version 0.220127.0436. The paper cites no DANDI version: its data statement points to the curated
epochs on Figshare, and the dandiset was published later. This is the only published version, and
the draft lists the same 55 files with the same total size, so the published version is pinned.
The files are streamed; nothing is downloaded.

**What the dataset holds.** 12 subjects and 55 subject-days of preprocessed ECoG (500 Hz), pose of
nine key points (30 frames per second), 8088 contralateral wrist movement events with the paper's
ten features, coarse behaviour labels, and an electrode table with MNI positions, a `good` flag and
the paper's regression R2 per electrode. Sex and age are not in it.

**Outside input.** The projection of electrodes onto the eight AAL regions is the paper's own
output on Figshare ([doi:10.6084/m9.figshare.12114408.v1](https://doi.org/10.6084/m9.figshare.12114408.v1)),
used for the region claims (`data: with_inputs`) and checked by the link claim `roi-projection-link`.

**Event sets.** Reading a is every event in the files. Reading b drops ECoG segments with missing
samples, with power below 0 dB or with 115-125 Hz power more than 3 SD above the subject's segments
(Methods); the summary of segment power these tests use is an assumption (helper docstring). The
false positives the paper removed are not marked in the files, so they cannot be dropped.

| Claim | Tested here |
|---|---|
| subject-days, roi-projection-link (link claims) | tested |
| cohort, fig2-event-counts, fig2-features-differ-across-subjects, fig2-reach-shape, fig2-bimanual-skew | tested |
| extfig2-1c-contralateral-larger, extfig2-2-feature-correlations, fig2-days-and-waking | tested |
| fig3-group-lfb-decrease-hfb-increase, fig3-sensorimotor-largest, fig3-lfb-more-widespread | tested (the last blocked: its readings disagree) |
| fig4-precentral-subject-variability, fig5-band-power-differs-across-subjects, table1-subject-band-power-ci | tested |
| fig5-recording-day-effects, extfig5-1-baseline-reduces-day-variability, discussion-day-changes-small | tested |
| fig6b-best-models-sensorimotor, fig6-r2-at-most-025, fig6c-angle-and-day-most-informative, extfig6-2-angle-most-retained, fig6d-upward-reaches-enhance-pattern, fig6d-angle-and-day-largest-coefficients | tested |

The notebook reads about 8000 ECoG segments of 10 s from 55 files of 8 to 20 GB each; that step took
20 to 35 minutes in recent runs (the time is printed below). The regression models take a few minutes.

In [ ]:
import os
import time
import warnings

os.environ["OMP_NUM_THREADS"] = "1"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ajile

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
SUBJECTS = [f"{i:02d}" for i in range(1, 13)]

In [ ]:
started = time.time()
assets = ajile.list_assets()
sessions = ajile.load_sessions(assets)
# LOAD-BEH-END
print(f"{len(sessions)} files of dandiset {ajile.DANDISET_ID} version {ajile.DANDISET_VERSION}, "
      f"{assets.size_gb.sum():.0f} GB in all; metadata, events and wrist traces read in "
      f"{time.time() - started:.0f} s")

In [ ]:
# One table of events. Time of day from the midnight clock of the files (see the helper).
rows = []
for s in sessions:
    e = s["events"].copy()
    e["subject"], e["day"], e["k"] = s["subject"], s["day"], np.arange(len(e))
    rows.append(e)
ev = pd.concat(rows, ignore_index=True)
ev = ev.rename(columns={"Reach_magnitude_px": "magnitude", "Reach_angle_degrees": "angle_file",
                        "Onset_speed_px_per_sec": "speed", "Speech_ratio": "speech",
                        "Bimanual_ratio": "bimanual_ratio", "Bimanual_overlap": "bimanual_overlap",
                        "Bimanual_class": "bimanual_class"})
# The files' angle is atan2(dy, dx) in image coordinates (checked below); the paper's angle
# negates it (y up) and folds 90-270 degrees onto 90 to -90 (Methods; code).
ev["angle"] = ajile.fold_angle(-ev["angle_file"])
ev["hour"] = ev["start_time"] / 3600.0
ev["tod_bin"] = np.clip((ev["hour"] // 8).astype(int), 0, 2)
FEATURES = ["day", "hour", "duration", "magnitude", "angle", "speed", "speech",
            "bimanual_ratio", "bimanual_overlap", "bimanual_class"]
print(len(ev), "events")

## Data check

Counts by subject and day, units and ranges of every variable used, and the clocks of the
data streams. Each check of the metadata against the data is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "subject": s["subject"], "day": s["day"], "subject_id": s["subject_id"], "session_id": s["session_id"],
    "session_start_time": s["session_start_time"], "electrodes": len(s["electrodes"]),
    "good": int(s["electrodes"]["good"].sum()), "events": len(s["events"]), "wrist": s["reach_event_desc"],
    "ecog_hours": round(s["ecog_shape"][0] / s["ecog_rate"] / 3600, 2), "ecog_Hz": s["ecog_rate"],
    "pose_hours": round(s["pose_shape"][0] / s["pose_rate"] / 3600, 2), "pose_Hz": s["pose_rate"],
    "first_event_h": round(s["events"].start_time.min() / 3600, 2),
    "last_event_h": round(s["events"].start_time.max() / 3600, 2)} for s in sessions])
print(info.to_string(index=False))
print("\nSubjects:", info.subject.nunique(), "; files:", len(info), "; events:", int(info.events.sum()))
print("subject_id and session_id agree with the file names:",
      bool((info.subject_id == info.subject).all() and (info.session_id.astype(int) == info.day).all()))
print("Subject fields in the files:", sorted({tuple(s["subject_fields"]) for s in sessions}))
print("ECoG unit and conversion:", {(s["ecog_unit"], s["ecog_conversion"]) for s in sessions},
      "; pose unit:", {s["pose_unit"] for s in sessions})
LOC = ["x", "y", "z", "group_name", "low_freq_R2", "high_freq_R2"]
print("Electrode positions, groups and stored R2 identical across a subject's files:",
      all(sessions[i]["electrodes"][LOC].equals(sessions[j]["electrodes"][LOC])
          for i in range(len(sessions)) for j in range(len(sessions))
          if sessions[i]["subject"] == sessions[j]["subject"]))
print("Good flags differ between a subject's files:",
      sorted({s["subject"] for s in sessions for t in sessions
              if s["subject"] == t["subject"] and not np.array_equal(s["electrodes"].good.values, t["electrodes"].good.values)}))

In [ ]:
# Columns of the reaches table in every file: is any event marked as a false positive or verified?
rc = pd.Series({f"S{s['subject']} day {s['day']}": ", ".join(s["reach_columns"]) for s in sessions})
for cols_, files in rc.groupby(rc).groups.items():
    print(f"{len(files)} files: {cols_}")
print("Any column naming false positives, verification or removal:",
      [c for s in sessions for c in s["reach_columns"] if any(w in c.lower() for w in ["false", "verif", "remov", "valid", "fp"])])

In [ ]:
# Electrode groups and their descriptions in each subject's first file, beside the data
# descriptor's Table 2 (Peterson et al. 2022, Sci Data 9:184, doi:10.1038/s41597-022-01280-y,
# "Surface electrodes: # good/total", "Depth electrodes: # good/total"; totals given here).
DESC_T2 = {"01": (86, 8), "02": (70, 16), "03": (80, 16), "04": (84, 0), "05": (106, 0), "06": (80, 0),
           "07": (64, 0), "08": (92, 0), "09": (98, 28), "10": (86, 40), "11": (106, 0), "12": (92, 32)}
for sub in SUBJECTS:
    s = [x for x in sessions if x["subject"] == sub][0]
    n = s["electrodes"].group_name.value_counts()
    print(f"S{sub} ({len(s['electrodes'])} electrodes; descriptor {DESC_T2[sub][0]} surface + {DESC_T2[sub][1]} depth): "
          + "; ".join(f"{g} {int(n.get(g, 0))} '{d}'" for g, d in s["group_desc"].items()))

The files hold 12 subjects and 55 subject-days, ECoG at 500 Hz and pose at 30 frames per
second. The ECoG is stored with a conversion of 1e-6 to volts, so the stored values are
microvolts; the pose is labelled meters but holds pixels (the data descriptor gives pixels, and
the values span hundreds). The subject has no sex or age in the files or the archive's metadata.
Three files hold less than 24 h of ECoG and start later (S01 day 7, S04 day 3, S06 day 3); the
clocks of these files are checked below.

The reaches table has the same nine columns in every file, and none marks false positives or
verified events. The electrode groups whose description says depth are listed above beside the
data descriptor's Table 2; S10 and S11 are counted differently there (see claim `cohort`).

In [ ]:
desc = {c: None for c in ["start_time", "stop_time", "duration", "magnitude", "angle_file", "angle", "speed",
                          "speech", "bimanual_ratio", "bimanual_overlap", "bimanual_class", "hour"]}
print(ev[list(desc)].describe().T[["count", "mean", "min", "50%", "max"]].round(3).to_string())
print("\nMissing values in any feature:", int(ev[list(desc)].isna().sum().sum()))
print("Bimanual ratio exactly 0.5 (the code's fill for missing values):", int((ev.bimanual_ratio == 0.5).sum()))
print("Bimanual overlap above 1 (the file calls it a fraction of the movement):",
      int((ev.bimanual_overlap > 1).sum()), "events")
print("Bimanual class values:", sorted(ev.bimanual_class.unique()))
print("Events per day, largest:", int(ev.groupby(["subject", "day"]).size().max()),
      "(Methods: up to 200 per day)")
print("Reach durations (stop - start) within 0.5-4 s (Methods' movement duration rule):",
      int(ev.duration.between(0.5, 4).sum()), "of", len(ev), "; shortest", round(ev.duration.min(), 3), "s")

In [ ]:
# Reach angle convention: angle and magnitude from the wrist trace (onset to the largest
# displacement within the reach), compared with the files' values.
rows = []
for s in sessions:
    w = "R_Wrist" if s["reach_event_desc"] == "r_wrist" else "L_Wrist"
    tr = s["wrist"][w]
    for k, d in enumerate(s["events"].duration.values):
        n = int(round(d * ajile.POSE_HZ))
        seg = tr[k, 45:46 + min(n, 45)]
        if np.isnan(seg).any():
            continue
        dd = seg - seg[0]
        r = np.linalg.norm(dd, axis=1)
        j = r.argmax()
        rows.append((s["events"]["Reach_angle_degrees"].values[k], np.degrees(np.arctan2(dd[j, 1], dd[j, 0])),
                     s["events"]["Reach_magnitude_px"].values[k], r[j], d <= 1.5))
ang = pd.DataFrame(rows, columns=["file_angle", "trace_angle", "file_mag", "trace_mag", "within_window"])
circ = lambda a, b: np.abs((a - b + 180) % 360 - 180)
a = ang[ang.within_window]
print(f"{len(a)} reaches of at most 1.5 s with a full trace")
print("median |file angle - trace angle|, image coordinates:", round(np.median(circ(a.file_angle, a.trace_angle)), 2),
      "deg; with the trace angle negated (y up):", round(np.median(circ(a.file_angle, -a.trace_angle)), 2), "deg")
print("median file magnitude / trace magnitude:", round(np.median(a.file_mag / a.trace_mag), 3))
print("Folded angle (paper convention, +90 up): share of events above 0:", round((ev.angle > 0).mean(), 3))

The files' reach angle is the angle of the reach in image coordinates (y pointing down), not yet
negated, so upward reaches have negative values; the paper's angle (+90 for upward) is its
negation folded onto -90 to 90, as the code does. The files do not say which convention they use.
The magnitudes match the traces.

In [ ]:
# Clock of the pose and of the event times: contralateral wrist displacement from its mean
# position over -1.5 to -1 s, at the events and at the events shifted by -30 s.
def median_disp(tr, base_slice=slice(0, 16)):
    base = np.nanmean(tr[:, base_slice], 1)
    return np.nanmedian(np.linalg.norm(tr - base[:, None], axis=2), 0)

rows = []
for s in sessions:
    w = "R_Wrist" if s["reach_event_desc"] == "r_wrist" else "L_Wrist"
    m = median_disp(s["wrist"][w])
    rows.append({"subject": s["subject"], "day": s["day"], "disp_at_-1s": m[15].round(1),
                 "disp_at_0s": m[45].round(1), "disp_at_+1s": m[75].round(1)})
clk = pd.DataFrame(rows)
print(clk.to_string(index=False))
print("Sessions where the contralateral wrist moves less than 10 pixels by +1 s:",
      clk[clk["disp_at_+1s"] < 10][["subject", "day"]].values.tolist())

In every file the contralateral wrist is still before the event time and moves after it, so the
pose and the event table share a clock, including in the three files with less than 24 h of
ECoG (whose events lie between 8 and 24 h). The shifted control needs the ECoG and is below.

In [ ]:
# Time of day: the coarse behaviour labels by hour. Sleep should fall at night if the files
# start at midnight, as the data descriptor says.
lab = pd.concat([s["epochs"].assign(subject=s["subject"]) for s in sessions])
lab["hour"] = ((lab.start + lab.stop) / 2 / 3600).astype(int)
sleep = lab[lab.label == "Sleep/rest"].groupby("hour").size().reindex(range(24), fill_value=0)
print("Sleep/rest labels by hour of the file clock:\n", sleep.to_string())
print("Share of Sleep/rest labels at 0-8 h:", round(sleep[:8].sum() / sleep.sum(), 3),
      "; events at 0-8 h:", round((ev.hour < 8).mean(), 3))
print("session_start_time values:", sorted(info.session_start_time.unique()))

Sleep labels fall mostly in the first hours of the file clock, which fits files that start at
midnight local time. The files' `session_start_time` of 19:00 at UTC-5 is midnight UTC, so the
stated time zone is not the local one; the three late files have start times of 03:07 to 03:31
at UTC-5, the start of their ECoG (checked below). The hour of an event is taken from the file
clock.

In [ ]:
# Duplicates: identical event tables or wrist traces in two files.
keys = {}
for s in sessions:
    k1 = np.round(s["events"].start_time.values, 3).tobytes()
    k2 = np.nan_to_num(s["wrist"]["R_Wrist"]).round(2).tobytes()
    keys.setdefault(("events", hash(k1)), []).append((s["subject"], s["day"]))
    keys.setdefault(("wrist", hash(k2)), []).append((s["subject"], s["day"]))
dups = {k: v for k, v in keys.items() if len(v) > 1}
print("Files with identical event times or identical wrist traces:", dups if dups else "none")

## ECoG: projection matrices and event segments

The paper's projection of electrodes onto the eight regions (AAL atlas through the Measure
Projection Toolbox) is taken from its own output on Figshare (doi:10.6084/m9.figshare.12114408.v1),
which the claims file names as the paper's output of that step. Claim `roi-projection-link`
checks that its rows can be matched to the dataset's electrodes. The bad electrodes are those of
each subject's first file, as the paper's code takes them.

In [ ]:
roi_meta = ajile.roi_files()
roi_mats, weights_check = {}, {}
for sub in SUBJECTS:
    roi_mats[sub] = ajile.read_roi_matrix(sub, roi_meta)
first_good = {}
for s in sessions:
    first_good.setdefault(s["subject"], s["electrodes"]["good"].values.copy())
for s in sessions:
    s["subject_good"] = first_good[s["subject"]]
print("Figshare files read (name, md5):")
for sub in SUBJECTS:
    print(f"  aal_{sub}_elecs2ROI.csv", roi_meta[f"aal_{sub}_elecs2ROI.csv"][1])

In [ ]:
started = time.time()
weights = {sub: ajile.roi_weights(roi_mats[sub], first_good[sub]) for sub in SUBJECTS}
ecog = ajile.load_ecog(ajile.ecog_jobs(sessions, weights))
# LOAD-ECOG-END
print(f"ECoG segments of {sum(sum(r['ok']) for r in ecog)} events read and reduced in "
      f"{(time.time() - started) / 60:.1f} min")

In [ ]:
# Per-event ECoG measures, aligned with the event table.
REG = {n: i for i, n in enumerate(ajile.ROI_NAMES)}
ev["ok"] = np.concatenate([r["ok"] for r in ecog])
for key in ["q_all", "q_hf", "q_log"]:
    ev[key] = np.concatenate([r[key] for r in ecog])
spec = np.full((len(ev), 8, len(ajile.FREQS), len(ajile.TIMES)), np.nan, dtype=np.float32)
i = 0
for r in ecog:
    for ok, sp in zip(r["ok"], r["roi_spec"]):
        if ok:
            spec[i] = sp
        i += 1
for kind in ["roi_band", "roi_band_raw"]:
    for meas in ["lfb_mean", "lfb_median", "hfb_mean", "hfb_median"]:
        vals = np.full((len(ev), 8), np.nan)
        i = 0
        for r in ecog:
            for ok, b in zip(r["ok"], r[kind]):
                if ok:
                    vals[i] = b[meas]
                i += 1
        for reg, j in REG.items():
            ev[f"{kind}:{reg}:{meas}"] = vals[:, j]
# Bad segments (Methods): power below 0 dB, or 115-125 Hz power more than 3 SD above the mean
# of the subject's segments. False positives are not marked in the files (see the data check).
thr = ev[ev.ok].groupby("subject").q_hf.agg(lambda v: v.mean() + 3 * v.std())
ev["bad_missing"] = ev.ok & ~np.isfinite(ev.q_all)          # segments with missing (NaN) samples
ev["bad_low"] = ev.q_all < 0
ev["bad_hf"] = ev.q_hf > ev.subject.map(thr)
ev["keep_b"] = ev.ok & ~ev.bad_missing & ~ev.bad_low & ~ev.bad_hf
TIMES, FREQS = ajile.TIMES, ajile.FREQS
print("Events with a full ECoG segment:", int(ev.ok.sum()), "of", len(ev))
print("Segments with missing (NaN) samples:", int(ev.bad_missing.sum()), "in",
      ev[ev.bad_missing].groupby(["subject", "day"]).size().to_dict())
print("Segments below 0 dB:", int(ev.bad_low.sum()), "; 115-125 Hz above 3 SD:", int((ev.bad_hf & ~ev.bad_low).sum()),
      "; kept (reading b):", int(ev.keep_b.sum()))
print("Other reading of the 0 dB test (mean of the log density below 0 dB):", int((ev.q_log < 0).sum()), "of",
      int(ev.ok.sum()), "segments would go, so that reading is not usable")
print(ev[ev.ok].groupby("subject")[["q_all", "q_hf"]].agg(["min", "median", "max"]).round(1).to_string())

In [ ]:
# segment_power against MNE's own Morlet transform, on one segment of the data.
from mne.time_frequency import tfr_array_morlet
import h5py, remfile
s0 = sessions[0]
f0 = h5py.File(remfile.File(ajile.asset_url(s0["asset_id"])), "r")
c0 = int(round(s0["events"].start_time.values[0] * ajile.FS))
x0 = f0["acquisition/ElectricalSeries/data"][c0 - 2500:c0 + 2500, :8].T.astype(float)
f0.close()
mine = ajile.segment_power(x0)
ref = tfr_array_morlet(x0[None], sfreq=ajile.FS, freqs=FREQS, n_cycles=FREQS / 4, zero_mean=True, use_fft=False,
                       decim=ajile.DECIM, output="power", verbose=False)[0][..., 5:96]
print("largest difference from MNE tfr_array_morlet(use_fft=False), dB:", float(np.abs(mine - 10 * np.log10(ref)).max()))
print("segment amplitude range (microvolts):", round(float(x0.min()), 1), round(float(x0.max()), 1))

In [ ]:
# Clock of the ECoG against the events: precentral and postcentral band power change from the
# -1.5 to -1 s baseline to 0-0.5 s, at the event time and at a pseudo-event 3 s earlier
# (-3 to -2.5 s against -4.5 to -4 s) in the same segments; median over events per file.
ti = lambda lo, hi: np.nonzero((TIMES >= lo - 1e-6) & (TIMES <= hi + 1e-6))[0]
lf = np.nonzero((FREQS >= 8) & (FREQS <= 32))[0]
hf = np.nonzero((FREQS >= 76) & (FREQS <= 100))[0]
def band_change(sp, band, t_on, t_base):
    return sp[..., band, :][..., t_on].mean((-1, -2)) - sp[..., band, :][..., t_base].mean((-1, -2))
rows = []
for (sub, day), g in ev[ev.keep_b].groupby(["subject", "day"]):
    sp = spec[g.index][:, [REG["precentral"], REG["postcentral"]]]
    row = {"subject": sub, "day": day, "n": len(g)}
    for name, band in [("HFB", hf), ("LFB", lf)]:
        true = band_change(sp, band, ti(0, 0.5), ti(-1.5, -1))
        pseudo = band_change(sp, band, ti(-3, -2.5), ti(-4.5, -4))
        best = np.argmax(np.abs(np.median(true, 0)))
        row[f"{name}_event"] = round(float(np.median(true[:, best])), 2)
        row[f"{name}_pseudo"] = round(float(np.median(pseudo[:, best])), 2)
    rows.append(row)
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))
clock_ok = (clock.HFB_event > 0) & (clock.LFB_event < 0) & (clock.HFB_event > clock.HFB_pseudo) & (clock.LFB_event < clock.LFB_pseudo)
print("Files with HFB up, LFB down at the event and both larger than at the pseudo-event:", int(clock_ok.sum()), "of", len(clock))
print("Files where this does not hold:", clock[~clock_ok][["subject", "day"]].values.tolist())

In [ ]:
# The three files with less than 24 h of ECoG: event times on the ECoG clock with the offset
# of the helper (ECoG ends at midnight) against no offset (ECoG starts at midnight).
late = [s for s in sessions if ajile.ecog_offset(s) > 1]
alt = ajile.load_ecog(ajile.ecog_jobs(late, weights, offset=False), workers=3)
for s, r in zip(late, alt):
    g = ev[(ev.subject == s["subject"]) & (ev.day == s["day"]) & ev.keep_b]
    pre = REG["precentral"]
    a_h = band_change(spec[g.index][:, pre], hf, ti(0, 0.5), ti(-1.5, -1))
    a_l = band_change(spec[g.index][:, pre], lf, ti(0, 0.5), ti(-1.5, -1))
    sp_b = np.array([x[pre] for x, ok in zip(r["roi_spec"], r["ok"]) if ok and np.isfinite(x[pre]).all()])
    b_h = band_change(sp_b, hf, ti(0, 0.5), ti(-1.5, -1)) if len(sp_b) else np.array([np.nan])
    b_l = band_change(sp_b, lf, ti(0, 0.5), ti(-1.5, -1)) if len(sp_b) else np.array([np.nan])
    print(f"S{s['subject']} day {s['day']}: offset {ajile.ecog_offset(s):.1f} s; with offset {len(a_h)} events, "
          f"precentral HFB {np.median(a_h):+.2f} dB, LFB {np.median(a_l):+.2f} dB; without offset {len(sp_b)} events in "
          f"range, HFB {np.median(b_h):+.2f} dB, LFB {np.median(b_l):+.2f} dB; ReachEvents minus offset-corrected "
          f"times, max |diff| {np.max(np.abs(s['reach_event_times'] - (s['events'].start_time.values - ajile.ecog_offset(s)))):.3f} s")

## Claim `subject-days`

> The dataset's subjects and recording days are the paper's subjects S01 to S12 and the
> recording days the paper lists for each.

In [ ]:
TABLE2 = {"01": [3, 4, 5, 7], "02": [3, 4, 5, 6], "03": [3, 4, 5, 6], "05": [3, 4, 7]}
TABLE2 = {s: TABLE2.get(s, [3, 4, 5, 6, 7]) for s in SUBJECTS}
days = info.groupby("subject").day.apply(list).to_dict()
for sub in SUBJECTS:
    print(f"S{sub}: dataset days {days[sub]}, Table 2 {TABLE2[sub]}, match {days[sub] == TABLE2[sub]}")
print("All 12 match:", all(days[s] == TABLE2[s] for s in SUBJECTS), "; subject-days:", sum(len(v) for v in days.values()))

**Result: reproduced.** The dataset's subjects 01 to 12 have exactly the days of Table 2 (55 subject-days), so the subjects are matched by number.

## Claim `roi-projection-link`

> The rows of the paper's projection matrices (Figshare) can be matched one to one to the
> dataset's electrodes, and the weights fit the electrodes' positions.

In [ ]:
cent_rows = {}
rows = []
rng_link = np.random.default_rng(55)
for sub in SUBJECTS:
    df = roi_mats[sub]
    el = [s for s in sessions if s["subject"] == sub][0]["electrodes"]
    n_rows = len(df) - 1
    zero = np.nonzero(df.iloc[1:].transpose().mean().values == 0)[0]
    cen = pd.read_csv(roi_meta[f"aal_{sub}_ROIcentroids_Lside.csv"][0])
    cen = cen.iloc[[df.columns.get_loc(c) for c in ajile.ROI_COLUMNS]].to_numpy(float)
    w = weights[sub]
    xyz = el[["x", "y", "z"]].to_numpy(float).copy()
    xyz[:, 0] = -np.abs(xyz[:, 0])                       # right hemisphere mirrored to the left
    dist = np.linalg.norm(xyz[:, None, :] - cen[None], axis=2)
    use = w.sum(1) > 0
    agree = np.mean(np.argmax(w[use], 1) == np.argmin(dist[use], 1))
    null = np.mean([np.mean(np.argmax(w[use], 1) == np.argmin(dist[use][rng_link.permutation(use.sum())], 1))
                    for _ in range(200)])
    rho = stats.spearmanr(w[use].ravel(), -dist[use].ravel())[0]
    rows.append({"subject": sub, "electrodes": len(el), "matrix_rows": n_rows, "all_zero_rows": len(zero),
                 "zero_row_groups": ",".join(sorted(set(el.group_name.values[zero]))) if n_rows == len(el) else
                 f"rows {zero.min()}-{zero.max()} (beyond the {len(el)} electrodes)",
                 "rows_used": len(w), "top_weight_is_nearest_centroid": round(agree, 3),
                 "same_with_order_shuffled": round(null, 3), "spearman_weight_vs_proximity": round(rho, 3)})
link = pd.DataFrame(rows)
print(link.to_string(index=False))
print("Subjects whose rows match the electrodes one to one:", int((link.rows_used == link.electrodes).sum()), "of 12")
print("Subjects where the weights fit positions better than with the order shuffled:",
      int((link.top_weight_is_nearest_centroid > link.same_with_order_shuffled).sum()), "of 12")
dens = pd.concat([roi_mats[s].iloc[0] for s in SUBJECTS], axis=1).mean(1)
print("Regions with mean electrode density > 3 (Methods):",
      [c for c in dens.index if dens[c] > 3], "\nthe eight of the claims file:", ajile.ROI_COLUMNS)

**Result: reproduced.** For all 12 subjects the projection matrix has one row per electrode of the NWB file (S01 has four extra all-zero rows at the end, dropped by the code's rule), the all-zero rows of S03 and S12 fall on depth electrode groups (LAD and LPD; AID), and the region with the largest weight is the region with the nearest centroid for 0.59 to 0.79 of electrodes, against 0.16 to 0.26 with the electrode order shuffled; the Spearman correlation of weight with proximity is 0.92 to 0.94. The regions with mean density above 3 are the eight of the claims file.

## Claim `cohort`

> Twelve subjects (8 male, 4 female, 29.4 +- 7.9 years) had on average 87.0 +- 12.9 surface
> electrodes; five also had 23.2 +- 12.1 depth electrodes; five were implanted in the right
> hemisphere and seven in the left.

In [ ]:
rows = []
for sub in SUBJECTS:
    s = [x for x in sessions if x["subject"] == sub][0]
    el = s["electrodes"]
    depth_groups = [g for g, d in s["group_desc"].items() if "depth" in d.lower()]
    is_depth = el.group_name.isin(depth_groups)
    rows.append({"subject": sub, "surface": int((~is_depth).sum()), "depth": int(is_depth.sum()),
                 "depth_groups": ",".join(depth_groups), "wrist": s["reach_event_desc"],
                 "mean_x": round(el.x.mean(), 1)})
coh = pd.DataFrame(rows)
print(coh.to_string(index=False))
hemi_right = (coh.wrist == "l_wrist").sum()
print(f"\nsubjects {len(coh)}; surface electrodes mean {coh.surface.mean():.1f} +- {coh.surface.std():.1f} (SD); "
      f"subjects with depth electrodes {(coh.depth > 0).sum()}, their mean {coh[coh.depth > 0].depth.mean():.1f} +- "
      f"{coh[coh.depth > 0].depth.std():.1f}; right hemisphere (left wrist events, x > 0) {hemi_right}")
print("Right hemisphere by wrist agrees with the sign of mean x:", bool(((coh.wrist == "l_wrist") == (coh.mean_x > 0)).all()))
print("Sex and age: not in the files or the archive's metadata (subject fields:", sessions[0]["subject_fields"], ")")
both = coh.copy()
both["all"] = both.surface + both.depth
print("Variant, every electrode counted: mean", round(both["all"].mean(), 1))
d2 = pd.DataFrame(DESC_T2, index=["surface", "depth"]).T
print("Variant, the data descriptor's Table 2 counts: surface mean", round(d2.surface.mean(), 1), "+-",
      round(d2.surface.std(), 1), "; subjects with depth electrodes", int((d2.depth > 0).sum()), ", their mean",
      round(d2[d2.depth > 0].depth.mean(), 1), "+-", round(d2[d2.depth > 0].depth.std(), 1))
print("Subjects whose file counts differ from the descriptor's:",
      {s: (int(r.surface), int(r.depth), DESC_T2[s]) for s, r in coh.set_index("subject").iterrows() if (r.surface, r.depth) != DESC_T2[s]})


**Result: partially reproduced.** 12 subjects, a mean of 84.8 surface electrodes (87.0 +- 12.9), a mean of 23.7 depth electrodes in the subjects that have them (23.2 +- 12.1) and 5 right-hemisphere subjects match. Seven subjects have electrode groups described as depth in the files, where the paper says five. Sex and age are not in the dataset, so those two values are not testable. The data descriptor's Table 2 gives S10 86 surface and 40 depth electrodes and S11 106 surface and none, where the group descriptions in the files give 78 and 48, and 88 and 18. With the descriptor's counts the surface mean is 87.0 +- 12.9 and six subjects have depth electrodes (23.3 +- 12.0), still not five.

## Claim `fig2-event-counts`

> Each subject contributed between 151 and 947 movement events (median 640 across subjects).

In [ ]:
cnt = pd.DataFrame({"a_all_events": ev.groupby("subject").size(),
                    "b_bad_segments_dropped": ev[ev.keep_b].groupby("subject").size()})
print(cnt.T.to_string())
for c in cnt:
    v = cnt[c]
    print(f"{c}: fewest {v.min()} (151 +- 15.1), most {v.max()} (947 +- 94.7), median {v.median():.1f} (640 +- 64); "
          f"within: {abs(v.min() - 151) <= 15.1}, {abs(v.max() - 947) <= 94.7}, {abs(v.median() - 640) <= 64}")
print("Total events:", len(ev), "; the data descriptor (Peterson et al. 2022, Sci Data) states 6931 verified events")

**Result: not reproduced.** The dataset holds 211 to 996 events per subject (median 770.5); after dropping segments with missing samples, below 0 dB or with high 115-125 Hz power, 211 to 979 (median 739). The fewest (211 against 151 +- 15.1) and the median (against 640 +- 64) are outside tolerance under both readings; the most is within. The files hold 8088 events, where the data descriptor states 6931, and no event is marked as a false positive. The reaches table has the same nine columns in all 55 files and none marks an event as a false positive or as verified, so reading b cannot drop the false positives: the outcome rests on reading a and on reading b without that step.

## Claim `fig2-features-differ-across-subjects`

> All ten behavioural and environmental features differ significantly across subjects.

In [ ]:
rows = []
for label, sel in [("b", ev.keep_b), ("a", np.ones(len(ev), bool))]:
    for f in FEATURES:
        groups = [g[f].values for _, g in ev[sel].groupby("subject")]
        h, p = stats.kruskal(*groups)
        rows.append({"reading": label, "feature": f, "H": round(h, 1), "p": p, "p_below_0001": p < 0.001})
kw = pd.DataFrame(rows)
print(kw.to_string(index=False))
print("All ten p < 0.001, reading b:", bool(kw[kw.reading == "b"].p_below_0001.all()),
      "; reading a:", bool(kw[kw.reading == "a"].p_below_0001.all()))
print("Largest p:", kw.groupby("reading").p.max().to_dict())

**Result: reproduced.** All ten Kruskal-Wallis tests across subjects give p < 0.001 under both event sets (largest p 9.9e-35).

## Claim `fig2-reach-shape`

> Reach duration and reach magnitude vary little across subjects, with most reaches lasting
> under 2 s and covering under 200 pixels, and reach angle distributions tend to be bimodal with
> peaks near +90 and -90 degrees, with few side-to-side reaches.

In [ ]:
from scipy.stats import gaussian_kde
grid = np.linspace(-90, 90, 361)
def bimodal(a):
    dens = gaussian_kde(a, bw_method="scott")(grid)
    peaks = [i for i in range(1, len(grid) - 1) if dens[i] > dens[i - 1] and dens[i] >= dens[i + 1]]
    # ends of the range count as maxima when the density rises towards them
    if dens[0] > dens[1]:
        peaks.append(0)
    if dens[-1] > dens[-2]:
        peaks.append(len(grid) - 1)
    top = sorted(peaks, key=lambda i: -dens[i])[:2]
    locs = sorted(grid[top])
    zero = dens[np.argmin(np.abs(grid))]
    ok = len(locs) == 2 and abs(locs[0] + 90) <= 15 and abs(locs[1] - 90) <= 15 and zero < dens[top].min()
    return ok, locs, zero / dens[top].min() if len(top) else np.nan
E = ev[ev.keep_b]
rows = []
for sub, g in list(E.groupby("subject")) + [("pooled", E)]:
    ok, locs, rel = bimodal(g.angle.values)
    rows.append({"subject": sub, "n": len(g), "median_duration_s": round(g.duration.median(), 2),
                 "share_under_2s": round((g.duration < 2).mean(), 3), "median_magnitude_px": round(g.magnitude.median(), 1),
                 "share_under_200px": round((g.magnitude < 200).mean(), 3), "angle_peaks": np.round(locs, 1).tolist(),
                 "density_at_0_over_lower_peak": round(rel, 2), "bimodal_at_pm90": ok,
                 "share_within_30deg_of_0": round((g.angle.abs() < 30).mean(), 3)})
rs = pd.DataFrame(rows)
print(rs.to_string(index=False))
subj = rs[rs.subject != "pooled"]
pool = rs[rs.subject == "pooled"].iloc[0]
dm = (subj.share_under_2s > 0.5) & (subj.share_under_200px > 0.5)
print(f"\nduration and magnitude part: pooled {pool.share_under_2s > 0.5 and pool.share_under_200px > 0.5}, "
      f"subjects {int(dm.sum())} of 12")
print(f"angle part: pooled {pool.bimodal_at_pm90}, subjects {int(subj.bimodal_at_pm90.sum())} of 12")
print("Variant, reading a (all events):")
E2 = ev
print(" share under 2 s", round((E2.duration < 2).mean(), 3), "; under 200 px", round((E2.magnitude < 200).mean(), 3),
      "; pooled angle bimodal at +-90:", bimodal(E2.angle.values)[0], np.round(bimodal(E2.angle.values)[1], 1).tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for sub, g in E.groupby("subject"):
    axes[0].hist(g.duration, bins=np.linspace(0, 4, 41), histtype="step", density=True)
    axes[1].hist(g.magnitude, bins=np.linspace(0, 700, 71), histtype="step", density=True)
    axes[2].plot(grid, gaussian_kde(g.angle, bw_method="scott")(grid), lw=1)
axes[0].set_xlabel("reach duration (s)"); axes[1].set_xlabel("reach magnitude (pixels)"); axes[2].set_xlabel("reach angle (deg, +90 up)")
plt.tight_layout(); plt.show()
print("Plotted ranges: duration", round(E.duration.min(), 2), "-", round(E.duration.max(), 2), "s; magnitude",
      round(E.magnitude.min(), 1), "-", round(E.magnitude.max(), 1), "px (none outside the axes)")

In [ ]:
# Sensitivity, not the criterion: how far a Gaussian KDE without boundary correction pulls peaks
# in from the ends of a bounded range. Synthetic folded angles whose density peaks exactly at
# +-90 (90 minus a half-normal of SD 25 degrees, either sign), and the same KDE with the data
# reflected at -90 and +90 (a standard boundary correction).
rng_kde = np.random.default_rng(2)
def kde_peaks(a, reflect=False):
    data = np.r_[a, -180 - a, 180 - a] if reflect else a
    kde = gaussian_kde(data, bw_method="scott")
    if reflect:
        kde.set_bandwidth(kde.factor * (len(data) / len(a)) ** 0.2)   # Scott's bandwidth of the original n
    dens = kde(grid)
    peaks = [i for i in range(1, len(grid) - 1) if dens[i] > dens[i - 1] and dens[i] >= dens[i + 1]]
    peaks += [i for i, j in [(0, 1), (len(grid) - 1, len(grid) - 2)] if dens[i] > dens[j]]
    return sorted(float(v) for v in np.round(grid[sorted(peaks, key=lambda i: -dens[i])[:2]], 1))
for n in [250, 900]:
    syn = np.sign(rng_kde.random(n) - 0.5) * (90 - np.abs(rng_kde.normal(0, 25, n)))
    syn = np.clip(syn, -90, 90)
    print(f"synthetic, n = {n}: Scott KDE peaks {kde_peaks(syn)}; reflected KDE peaks {kde_peaks(syn, True)}")
rows = []
for sub, g in list(E.groupby("subject")) + [("pooled", E)]:
    pk = kde_peaks(g.angle.values, True)
    rows.append({"subject": sub, "criterion_KDE_peaks": kde_peaks(g.angle.values), "reflected_KDE_peaks": pk,
                 "reflected_within_15_of_pm90": len(pk) == 2 and pk[0] <= -75 and pk[1] >= 75})
kr = pd.DataFrame(rows)
print(kr.to_string(index=False))
print("Reflected KDE: two highest maxima within 15 degrees of -90 and +90 in",
      int(kr[kr.subject != "pooled"].reflected_within_15_of_pm90.sum()), "of 12 subjects; pooled",
      bool(kr[kr.subject == "pooled"].reflected_within_15_of_pm90.iloc[0]))

**Result: partially reproduced.** More than half of the reaches last under 2 s and cover under 200 pixels pooled (0.918 and 0.915) and in all 12 subjects. The angle part does not hold: the two highest maxima of the angle density lie at -71.5 and 75.0 degrees pooled, and within 15 degrees of +-90 in no subject (S02 and S04 have a single maximum, and in S03, S06 and S07 the density at 0 exceeds the lower peak); 22.9% of reaches lie within 30 degrees of horizontal. With a KDE reflected at -90 and +90 (a sensitivity output, not the criterion), the two highest maxima lie within 15 degrees of +-90 pooled and in 6 of 12 subjects, not more than half. On synthetic angles whose density peaks at +-90 the criterion's KDE puts the peaks at -72.5 and 70.0 (n = 250) and at -73.5 and 74.0 degrees (n = 900), so part of the shortfall comes from the KDE's boundary bias.

## Claim `fig2-bimanual-skew`

> Bimanual ratio and bimanual overlap are skewed toward unimanual movement of the contralateral
> arm, less so for subjects 02, 04 and 05, while the bimanual class is mostly bimanual.

In [ ]:
def bim_table(E):
    rows = []
    for sub, g in list(E.groupby("subject")) + [("pooled", E)]:
        rows.append({"subject": sub, "median_ratio": round(g.bimanual_ratio.median(), 3),
                     "ratio_skewed": g.bimanual_ratio.median() < 0.5,
                     "overlap_skewness": round(stats.skew(g.bimanual_overlap), 3),
                     "overlap_skewed": stats.skew(g.bimanual_overlap) > 0,
                     "share_class_bimanual": round(g.bimanual_class.mean(), 3),
                     "class_mostly_bimanual": g.bimanual_class.mean() > 0.5})
    return pd.DataFrame(rows)
for label, E_ in [("b", ev[ev.keep_b]), ("a", ev)]:
    bt = bim_table(E_)
    subj = bt[bt.subject != "pooled"]
    pool = bt[bt.subject == "pooled"].iloc[0]
    top_ratio = subj.sort_values("median_ratio", ascending=False).subject.head(5).tolist()
    low_skew = subj.sort_values("overlap_skewness").subject.head(5).tolist()
    named = all(s in top_ratio for s in ["02", "04", "05"]) and all(s in low_skew for s in ["02", "04", "05"])
    print(f"Reading {label}")
    print(bt.to_string(index=False))
    print(f"ratio part: pooled {pool.ratio_skewed}, subjects {int(subj.ratio_skewed.sum())}/12; overlap part: pooled "
          f"{pool.overlap_skewed}, subjects {int(subj.overlap_skewed.sum())}/12; class part: pooled {pool.class_mostly_bimanual}, "
          f"subjects {int(subj.class_mostly_bimanual.sum())}/12")
    print(f"five highest median ratios {top_ratio}; five lowest overlap skewness {low_skew}; S02, S04, S05 in both: {named}\n")

**Result: reproduced.** The median bimanual ratio is below 0.5 and the overlap skewness positive pooled and in all 12 subjects, the class is bimanual for more than half of the events pooled (0.725) and in all 12, and S02, S04 and S05 are among the five subjects with the highest median ratio and among the five with the lowest overlap skewness, under both event sets.

## Claim `extfig2-1c-contralateral-larger`

> Across all 12 subjects, the median displacement of the contralateral wrist after movement
> onset is substantially greater than that of the ipsilateral wrist, confirming that event
> detection mainly picked contralateral movements.

In [ ]:
t_pose = np.round(np.arange(91) / ajile.POSE_HZ + ajile.POSE_WIN[0], 3)
post = (t_pose >= 0) & (t_pose <= 1.5)
def disp(tr, reading):
    base = np.nanmean(tr[:, :16], 1) if reading == "a" else tr[:, 45]   # -1.5 to -1 s, or onset
    return np.linalg.norm(tr - base[:, None], axis=2)
rows = []
for sub in SUBJECTS:
    ss = [s for s in sessions if s["subject"] == sub]
    contra = "R_Wrist" if ss[0]["reach_event_desc"] == "r_wrist" else "L_Wrist"
    ipsi = "L_Wrist" if contra == "R_Wrist" else "R_Wrist"
    keep = np.concatenate([ev[(ev.subject == sub) & (ev.day == s["day"])].keep_b.values for s in ss])
    row = {"subject": sub, "contra": contra}
    for reading in ["a", "b"]:
        c = np.concatenate([disp(s["wrist"][contra], reading) for s in ss])[keep]
        i_ = np.concatenate([disp(s["wrist"][ipsi], reading) for s in ss])[keep]
        cm, im = np.nanmedian(c, 0)[post].max(), np.nanmedian(i_, 0)[post].max()
        row[f"{reading}_contra_peak_px"] = round(cm, 1)
        row[f"{reading}_ipsi_peak_px"] = round(im, 1)
        row[f"{reading}_ratio"] = round(cm / im, 2)
    rows.append(row)
cw = pd.DataFrame(rows)
print(cw.to_string(index=False))
for r in ["a", "b"]:
    print(f"reading {r}: contralateral peak larger in {(cw[f'{r}_contra_peak_px'] > cw[f'{r}_ipsi_peak_px']).sum()} of 12; "
          f"ratio min {cw[f'{r}_ratio'].min()}, median {cw[f'{r}_ratio'].median()}")

**Result: reproduced.** The peak of the median contralateral displacement exceeds the ipsilateral one in all 12 subjects under both readings of the rest position (ratios 1.12 to 37.8 under reading a, 1.08 to 42.0 under reading b). In S04 and S05 the two are close (ratios 1.08 to 1.32).

## Claim `extfig2-2-feature-correlations`

> Over all events, reach magnitude correlates with reach duration (r = 0.26) and onset speed
> (r = 0.56), reach duration with bimanual overlap (r = 0.48), and the bimanual features with
> one another (overlap and ratio 0.51, class and ratio 0.50, overlap and class 0.61).

In [ ]:
PAIRS = [("magnitude", "duration", 0.26), ("magnitude", "speed", 0.56), ("duration", "bimanual_overlap", 0.48),
         ("bimanual_overlap", "bimanual_ratio", 0.51), ("bimanual_class", "bimanual_ratio", 0.50),
         ("bimanual_overlap", "bimanual_class", 0.61)]
rows = []
for label, E_ in [("b", ev[ev.keep_b]), ("a", ev)]:
    corr = E_[FEATURES].corr()
    for x, y, paper in PAIRS:
        r = corr.loc[x, y]
        rows.append({"reading": label, "pair": f"{x} - {y}", "r": round(r, 3), "paper": paper,
                     "within_10pct": abs(r - paper) <= 0.1 * paper + 1e-9})
cr = pd.DataFrame(rows)
print(cr.to_string(index=False))
print("Within tolerance:", cr.groupby("reading").within_10pct.sum().to_dict(), "of 6")
print("\nPer subject, reading b (r):")
print(pd.DataFrame({f"{x}-{y}": ev[ev.keep_b].groupby("subject").apply(lambda g: g[x].corr(g[y])).round(2)
                    for x, y, _ in PAIRS}).to_string())

**Result: partially reproduced.** Five of the six correlations are within tolerance (0.604, 0.503, 0.502, 0.494, 0.599); reach magnitude and duration give r = 0.233, just below the tolerance of 0.26 +- 0.026, under both event sets (0.231 with all events).

## Claim `fig2-days-and-waking`

> Reach counts were fairly even across recording days except for subjects 04, 05 and 09, each
> with one day holding most events, and events occurred mostly during waking hours.

In [ ]:
for label, E_ in [("b", ev[ev.keep_b]), ("a", ev)]:
    per_day = E_.groupby(["subject", "day"]).size().unstack(fill_value=0)
    busiest = (per_day.max(1) / per_day.sum(1)).round(3)
    awake = E_.groupby("subject").hour.apply(lambda h: (h >= 8).mean()).round(3)
    out = pd.DataFrame({"busiest_day_share": busiest, "share_08_24h": awake})
    print(f"Reading {label}\n", per_day.to_string(), "\n", out.T.to_string())
    over = out.index[out.busiest_day_share > 0.5].tolist()
    print(f"subjects whose busiest day holds more than half: {over}; named 04, 05, 09; "
          f"waking part (more than half at 08-24 h) holds in {(out.share_08_24h > 0.5).sum()} of 12\n")

**Result: partially reproduced.** One day holds more than half of the events for S05 (0.605) and S09 (0.517) and for no other subject, but not for S04, whose busiest day holds 0.379. More than half of the events fall at 08 to 24 h in all 12 subjects (0.641 to 0.964), so the waking part holds.

## Claim `fig3-group-lfb-decrease-hfb-increase`

> At the group level, low-frequency power decreases and high-frequency power increases at
> movement initiation in several cortical regions, as in controlled movement experiments.

In [ ]:
rng_fig3 = np.random.default_rng(3)
keep = ev.keep_b.values
subj_med = np.stack([np.median(spec[np.nonzero(keep & (ev.subject.values == sub))[0]], axis=0) for sub in SUBJECTS])
base_i = ti(-1.5, -1)
group = subj_med.mean(0)
subj_c = subj_med - np.median(group[..., base_i], axis=-1, keepdims=True)[None]     # plot_specs_group
masked = np.stack([ajile.boot_sig_mask(np.moveaxis(subj_c[:, r], 0, -1), base_i, rng_fig3) for r in range(8)])
unmasked = subj_c.mean(0)
LF34 = np.nonzero((FREQS >= 4) & (FREQS <= 30))[0]
HF50 = np.nonzero((FREQS >= 50) & (FREQS <= 120))[0]
t_a, t_b = ti(0, 0.5), np.array([np.argmin(np.abs(TIMES))])
rows = []
for r, name in enumerate(ajile.ROI_NAMES):
    row = {"region": name}
    for lab, tt in [("a_0-0.5s", t_a), ("b_t0", t_b)]:
        row[f"{lab}_LF4-30"] = round(float(masked[r][LF34][:, tt].mean()), 3)
        row[f"{lab}_HF50-120"] = round(float(masked[r][HF50][:, tt].mean()), 3)
    row["unmasked_a_LF"] = round(float(unmasked[r][LF34][:, t_a].mean()), 3)
    row["unmasked_a_HF"] = round(float(unmasked[r][HF50][:, t_a].mean()), 3)
    row["significant_bins"] = int((masked[r] != 0).sum())
    rows.append(row)
f3 = pd.DataFrame(rows)
print(f3.to_string(index=False))
named = ["precentral", "postcentral", "inferior parietal"]
for lab in ["a_0-0.5s", "b_t0"]:
    holds = f3[f3.region.isin(named)].apply(lambda r: r[f"{lab}_LF4-30"] < 0 and r[f"{lab}_HF50-120"] > 0, axis=1)
    print(f"reading {lab}: pattern holds in {int(holds.sum())} of 3 named regions",
          dict(zip(named, holds.values)))
print("Regions with the pattern under reading a:",
      f3[(f3["a_0-0.5s_LF4-30"] < 0) & (f3["a_0-0.5s_HF50-120"] > 0)].region.tolist())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharex=True, sharey=True)
tsel = ti(-1.5, 1.5)
for r, ax in enumerate(axes.ravel()):
    im = ax.pcolormesh(TIMES[tsel], FREQS, masked[r][:, tsel], cmap="RdBu_r", vmin=-1, vmax=1, shading="nearest")
    ax.axvline(0, color="k", lw=1)
    ax.set_title(ajile.ROI_NAMES[r], fontsize=9)
axes[1, 0].set_xlabel("time (s)"); axes[1, 0].set_ylabel("frequency (Hz)")
fig.colorbar(im, ax=axes, label="dB (masked)")
plt.show()
print("masked values shown, -1.5 to 1.5 s: range", round(float(masked[..., tsel].min()), 2), "to",
      round(float(masked[..., tsel].max()), 2), "dB; values beyond +-1 dB:", int((np.abs(masked[..., tsel]) > 1).sum()))

**Result: reproduced.** After masking, the mean over 4-30 Hz is negative and over 50-120 Hz positive in precentral (-0.885 and 0.496 dB), postcentral (-0.890, 0.507) and inferior parietal (-0.663, 0.277) over 0 to 0.5 s, and also at the time bin of onset (reading b). Over 0 to 0.5 s, middle frontal, supramarginal and middle temporal show the same pattern (middle temporal only weakly, -0.005 and +0.003 dB); at the onset bin supramarginal's high-frequency value and middle temporal's low-frequency value are 0.000, so for those two it holds only over 0 to 0.5 s.

## Claim `fig3-sensorimotor-largest`

> Spectral power changes are largest in precentral, postcentral and inferior parietal regions,
> and the three temporal regions show minimal movement-related changes.

In [ ]:
temporal = ["superior temporal", "middle temporal", "inferior temporal"]
def check_rank(vals):
    order = list(pd.Series(vals).sort_values(ascending=False).index)
    top3 = set(order[:3]) == set(named)
    temp_below = all(vals[t] < min(vals[n] for n in named) for t in temporal)
    return order, top3, temp_below, sum(n in order[:3] for n in named)
res = {}
va = {n: float(np.abs(masked[r][:, t_a]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
vc = {n: float(np.abs(masked[r][:, ti(-0.5, 0.5)]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
vl = {n: float(np.abs(masked[r][LF34][:, t_a]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
vh = {n: float(np.abs(masked[r][HF50][:, t_a]).mean()) for r, n in enumerate(ajile.ROI_NAMES)}
print(pd.DataFrame({"a_all_freqs_0-0.5": va, "b_LF": vl, "b_HF": vh, "c_all_freqs_-0.5-0.5": vc}).round(3).to_string())
for lab, v in [("a", va), ("c", vc)]:
    order, top3, tb, k = check_rank(v)
    print(f"reading {lab}: order {order}; named three are the top three {top3} ({k} of 3); temporal below all three {tb}")
ol, tl, tbl, kl = check_rank(vl)
oh, th, tbh, kh = check_rank(vh)
print(f"reading b: LF order {ol}\n           HF order {oh}\n  top three in both {tl and th} (LF {kl}/3, HF {kh}/3); "
      f"temporal below all three in both {tbl and tbh}")

**Result: reproduced.** Under all three readings, postcentral, precentral and inferior parietal are the three regions with the largest mean absolute masked power, and the three temporal regions are below all of them (for example reading a: 0.523, 0.512, 0.336 dB against at most 0.039 for the temporal regions).

## Claim `fig3-lfb-more-widespread`

> Low-frequency power decreases are more spatially widespread than high-frequency increases and
> are also present in supramarginal and superior temporal regions.

In [ ]:
neg_lf = {n: int((masked[r][LF34][:, t_a] < 0).sum()) for r, n in enumerate(ajile.ROI_NAMES)}
pos_hf = {n: int((masked[r][HF50][:, t_a] > 0).sum()) for r, n in enumerate(ajile.ROI_NAMES)}
print(pd.DataFrame({"significant negative 4-30 Hz bins": neg_lf, "significant positive 50-120 Hz bins": pos_hf}).to_string())
a_lf, a_hf = sum(v > 0 for v in neg_lf.values()), sum(v > 0 for v in pos_hf.values())
b_lf, b_hf = sum(neg_lf.values()) / len(LF34), sum(pos_hf.values()) / len(HF50)
print(f"reading a: regions with LF decreases {a_lf}, with HF increases {a_hf}; LF larger {a_lf > a_hf}")
print(f"reading b: LF bins per frequency {b_lf:.2f}, HF bins per frequency {b_hf:.2f}; LF larger {b_lf > b_hf}")
print("supramarginal and superior temporal have LF decreases:", neg_lf["supramarginal"] > 0, neg_lf["superior temporal"] > 0)

In [ ]:
# Seed stability of the bootstrap mask: the same masks with ten other seeds.
rows = []
for seed in range(10):
    m_ = np.stack([ajile.boot_sig_mask(np.moveaxis(subj_c[:, r], 0, -1), base_i, np.random.default_rng(100 + seed))
                   for r in range(8)])
    nl = [int((m_[r][LF34][:, t_a] < 0).sum()) for r in range(8)]
    ph = [int((m_[r][HF50][:, t_a] > 0).sum()) for r in range(8)]
    pattern = [bool(m_[r][LF34][:, t_a].mean() < 0 and m_[r][HF50][:, t_a].mean() > 0) for r in range(8)]
    rows.append({"seed": 100 + seed, **{f"LF-:{n[:12]}": v for n, v in zip(ajile.ROI_NAMES, nl)},
                 **{f"HF+:{n[:12]}": v for n, v in zip(ajile.ROI_NAMES, ph)},
                 "a_LF_regions": sum(v > 0 for v in nl), "a_HF_regions": sum(v > 0 for v in ph),
                 "b_LF_per_f": round(sum(nl) / len(LF34), 2), "b_HF_per_f": round(sum(ph) / len(HF50), 2),
                 "fig3_pattern_in_named_3": sum(pattern[REG[n]] for n in named)})
sd = pd.DataFrame(rows)
print(sd.T.to_string())
print("Reading a, low-frequency regions more than high-frequency in", int((sd.a_LF_regions > sd.a_HF_regions).sum()),
      "of 10 seeds, equal in", int((sd.a_LF_regions == sd.a_HF_regions).sum()), "; reading b, low larger in",
      int((sd.b_LF_per_f > sd.b_HF_per_f).sum()), "of 10; supramarginal and superior temporal low-frequency decreases in",
      int(((sd["LF-:supramargina"] > 0) & (sd["LF-:superior tem"] > 0)).sum()), "of 10; fig3 pattern in all 3 named regions in",
      int((sd.fig3_pattern_in_named_3 == 3).sum()), "of 10")

**Result: blocked (methods underspecified).** The two readings disagree. Reading a counts 7 regions with low-frequency decreases and 7 with high-frequency increases (no difference); reading b gives 35.2 significant low-frequency bins per frequency against 25.9 high-frequency (low-frequency larger). Supramarginal and superior temporal both have significant 4-30 Hz decreases (30 and 24 bins). With ten other bootstrap seeds, reading a gives 7 against 7 regions every time and reading b 35.0 to 35.6 against 25.5 to 25.8 bins per frequency, so the readings disagree under every seed; supramarginal and superior temporal keep their low-frequency decreases and the pattern of fig3-group-lfb-decrease-hfb-increase holds in the three named regions under every seed.

## Claim `fig4-precentral-subject-variability`

> In the precentral region every subject shows the low/high-frequency pattern, but amplitudes
> and frequency bands vary; subjects 03, 06, 07, 08 and 11 have increases up to 120 Hz, subjects
> 09 and 12 mainly at 60-80 Hz, and subjects 04 and 08 have narrower low-frequency decreases.

In [ ]:
pre = REG["precentral"]
sp4 = subj_med[:, pre] - subj_med[:, pre][..., base_i].mean(-1, keepdims=True)     # plot_specs_single_region
spectrum = pd.DataFrame(sp4[..., t_a].mean(-1), index=SUBJECTS, columns=FREQS)
print("Precentral spectrum, mean over 0-0.5 s (dB):")
print(spectrum.round(2).to_string())
lfb_i, hfb_i = (FREQS >= 8) & (FREQS <= 32), (FREQS >= 76) & (FREQS <= 100)
p1 = (spectrum.loc[:, lfb_i].mean(1) < 0) & (spectrum.loc[:, hfb_i].mean(1) > 0)
print("\n(1) LFB mean < 0 and HFB mean > 0:", p1.to_dict(), "->", bool(p1.all()))
p2 = {s: bool(spectrum.loc[s, 116] > 0 and spectrum.loc[s, 121] > 0) for s in ["03", "06", "07", "08", "11"]}
print("(2) 116 and 121 Hz positive:", p2, "->", all(p2.values()))
p3 = {}
for s in ["09", "12"]:
    hi = spectrum.loc[s, FREQS >= 56]
    peak_f = hi.idxmax()
    p3[s] = (int(peak_f), round(hi.max(), 3), round(spectrum.loc[s, (FREQS >= 101)].mean(), 3),
             bool(61 <= peak_f <= 76 and spectrum.loc[s, FREQS >= 101].mean() < hi.max() / 2))
print("(3) peak frequency, peak, mean 101-121 Hz, holds:", p3, "->", all(v[-1] for v in p3.values()))
low = spectrum.loc[:, FREQS <= 46]
width = {s: int((low.loc[s] < low.loc[s].min() / 2).sum()) for s in SUBJECTS}
fewest = sorted(width, key=lambda s: (width[s], s))[:3]
ties = sorted(width.values())[2]
print("(4) width of the decrease (1-46 Hz bins below half the most negative value):", width)
print("    three fewest:", fewest, "; third-smallest width", ties, "shared by",
      [s for s in SUBJECTS if width[s] == ties], "; S04 and S08 among them:", "04" in fewest and "08" in fewest)
# The bin count ties four subjects at 3 bins. Tie-break by the width in Hz of the run of the
# spectrum below half its minimum, around the minimum, with linear interpolation between bins.
def width_hz(v, f):
    m = int(np.argmin(v)); h = v[m] / 2
    lo = m
    while lo > 0 and v[lo - 1] < h:
        lo -= 1
    hi = m
    while hi < len(v) - 1 and v[hi + 1] < h:
        hi += 1
    left = f[lo] if lo == 0 else f[lo - 1] + (h - v[lo - 1]) / (v[lo] - v[lo - 1]) * (f[lo] - f[lo - 1])
    right = f[hi] if hi == len(v) - 1 else f[hi] + (h - v[hi]) / (v[hi + 1] - v[hi]) * (f[hi + 1] - f[hi])
    return right - left
whz = {s: round(width_hz(low.loc[s].values.astype(float), FREQS[FREQS <= 46].astype(float)), 1) for s in SUBJECTS}
fewest_hz = sorted(whz, key=lambda s: whz[s])[:3]
print("    width in Hz (interpolated):", whz, "\n    three narrowest:", fewest_hz,
      "; S04 and S08 among them:", "04" in fewest_hz and "08" in fewest_hz)
print("Parts holding:", int(p1.all()) + int(all(p2.values())) + int(all(v[-1] for v in p3.values())) +
      int("04" in fewest and "08" in fewest), "of 4")

**Result: reproduced.** All four parts hold: every subject has a negative LFB and positive HFB mean in precentral; S03, S06, S07, S08 and S11 are positive at 116 and 121 Hz; S09 and S12 peak at 66 Hz with the 101-121 Hz mean below half the peak; S08 has the narrowest decrease (2 bins, 9.4 Hz). S04 ties with S03, S05 and S07 at 3 bins; by the interpolated width S04 is second narrowest (14.4 Hz), so it is among the three.

## Claim `fig5-band-power-differs-across-subjects`

> Precentral LFB and HFB power in the 0.5 s after movement onset differs significantly across
> subjects.

In [ ]:
E = ev[ev.keep_b]
for meas in ["mean", "median"]:
    for band in ["lfb", "hfb"]:
        col = f"roi_band:precentral:{band}_{meas}"
        h, p = stats.kruskal(*[g[col].values for _, g in E.groupby("subject")])
        print(f"band {meas}, {band.upper()}: Kruskal-Wallis H = {h:.1f}, p = {p:.3g}, n = {len(E)} events in 12 subjects")

**Result: reproduced.** Kruskal-Wallis across the 12 subjects gives p = 1.3e-163 (LFB) and 7.1e-264 (HFB) with the band mean, and p < 1e-150 with the code's median.

## Claim `table1-subject-band-power-ci`

> Each subject's precentral band power lies in the 95% bootstrap interval Table 1 gives; LFB
> power is below baseline and HFB power above it for most subjects.

In [ ]:
TABLE1 = {"01": [(-1.87, -1.29), (0.03, 0.23)], "02": [(-1.12, -0.31), (0.07, 0.64)], "03": [(-2.11, -1.73), (1.39, 1.67)],
          "04": [(-0.17, 0.01), (0.32, 0.52)], "05": [(-0.7, -0.38), (0.2, 0.48)], "06": [(-1.63, -1.11), (2.02, 2.37)],
          "07": [(-0.84, -0.51), (1.14, 1.34)], "08": [(-0.3, 0.04), (0.36, 0.63)], "09": [(-0.57, -0.08), (-0.11, 0.23)],
          "10": [(-0.87, -0.35), (0.01, 0.3)], "11": [(-2.3, -1.86), (0.09, 0.34)], "12": [(-0.66, -0.44), (0.06, 0.2)]}
rng_t1 = np.random.default_rng(1)
rows, inside = [], {}
for sub in SUBJECTS:
    g = E[E.subject == sub]
    for bi_, band in enumerate(["lfb", "hfb"]):
        lo, hi = TABLE1[sub][bi_]
        row = {"subject": sub, "band": band.upper(), "paper_ci": f"{lo} to {hi}"}
        for bm in ["mean", "median"]:
            v = g[f"roi_band:precentral:{band}_{bm}"].values
            for stat_name, fn in [("mean", np.mean), ("median", np.median)]:
                est = fn(v)
                key = f"{stat_name}_of_band{bm}"
                row[key] = round(est, 3)
                inside.setdefault(key, []).append(lo <= est <= hi)
        v = g[f"roi_band:precentral:{band}_mean"].values
        boot = np.array([np.mean(v[rng_t1.integers(0, len(v), len(v))]) for _ in range(5000)])
        row["bootstrap_ci_mean_of_bandmean"] = f"{np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}"
        rows.append(row)
t1 = pd.DataFrame(rows)
print(t1.to_string(index=False))
print("\nPoint estimates inside the paper's interval, of 24 subject and band pairs:",
      {k: int(sum(v)) for k, v in inside.items()})
lfb_neg = (t1[t1.band == "LFB"]["mean_of_bandmean"] < 0).sum()
hfb_pos = (t1[t1.band == "HFB"]["mean_of_bandmean"] > 0).sum()
print(f"LFB below baseline in {lfb_neg} of 12 subjects; HFB above in {hfb_pos} of 12 (mean of band mean)")

**Result: partially reproduced.** With the mean over events of the band mean, 23 of 24 subject and band values lie inside the paper's intervals; S06 HFB is 1.78 dB against 2.02 to 2.37. The other readings give 20 to 21 of 24. LFB is below baseline and HFB above it in all 12 subjects.

## Claim `fig5-recording-day-effects`

> Precentral band power differs significantly across recording days in LFB for subjects 03,
> 05, 06, 07, 08 and 11, and in HFB for subjects 03, 04, 07, 08 and 10, despite baseline
> subtraction.

In [ ]:
PAPER_SIG = {("03", "lfb"), ("05", "lfb"), ("06", "lfb"), ("07", "lfb"), ("08", "lfb"), ("11", "lfb"),
             ("03", "hfb"), ("04", "hfb"), ("07", "hfb"), ("08", "hfb"), ("10", "hfb")}
rows = []
for meas in ["mean", "median"]:
    for band in ["lfb", "hfb"]:
        for sub in SUBJECTS:
            g = E[E.subject == sub]
            h, p = stats.kruskal(*[d[f"roi_band:precentral:{band}_{meas}"].values for _, d in g.groupby("day")])
            rows.append({"measure": meas, "band": band.upper(), "subject": sub, "p": p, "significant": p < 0.05,
                         "paper_significant": (sub, band) in PAPER_SIG})
dayk = pd.DataFrame(rows)
dayk["agrees"] = dayk.significant == dayk.paper_significant
print(dayk.pivot_table(index=["band", "subject"], columns="measure", values="p").round(4).to_string())
print("\nAgreement with the paper's 24 subject and band pairs:", dayk.groupby("measure").agrees.sum().to_dict())
for meas in ["mean", "median"]:
    d = dayk[dayk.measure == meas]
    print(f"band {meas}: significant here but not in the paper {d[d.significant & ~d.paper_significant][['band','subject']].values.tolist()}; "
          f"in the paper but not here {d[~d.significant & d.paper_significant][['band','subject']].values.tolist()}")

**Result: reproduced.** With the band mean, the Kruskal-Wallis tests across days agree with the paper on all 24 subject and band pairs (the 11 significant pairs and the 13 others); with the code's band median, 23 of 24 (LFB S05 gives p = 0.088).

## Claim `extfig5-1-baseline-reduces-day-variability`

> Subtracting the -1.5 to -1 s baseline reduces the variability of band power across recording
> days, though clear variability remains.

In [ ]:
# Electrode band power (band mean), with and without baseline subtraction, for kept events.
el = {}
i = 0
for s, r in zip(sessions, ecog):
    for k, ok in enumerate(r["ok"]):
        if keep[i]:
            for kind in ["el_band", "el_band_raw"]:
                for band in ["lfb", "hfb"]:
                    el.setdefault((s["subject"], kind, band), []).append((s["day"], r[kind][k][f"{band}_mean"]))
        i += 1
rows_a, rows_b = [], []
for sub in SUBJECTS:
    good = first_good[sub]
    for band in ["lfb", "hfb"]:
        g = E[E.subject == sub]
        sd = {}
        for kind, col in [("with", f"roi_band:precentral:{band}_mean"), ("without", f"roi_band_raw:precentral:{band}_mean")]:
            sd[kind] = g.groupby("day")[col].mean().std()
        rows_a.append({"subject": sub, "band": band.upper(), "sd_with": round(sd["with"], 3), "sd_without": round(sd["without"], 3)})
        for kind in ["with", "without"]:
            items = el[(sub, "el_band" if kind == "with" else "el_band_raw", band)]
            days_ = np.array([d for d, _ in items])
            vals = np.array([v for _, v in items])[:, good]
            dm = np.array([vals[days_ == d].mean(0) for d in np.unique(days_)])
            sd[kind + "_el"] = dm.std(0, ddof=1)
        for k_ in range(good.sum()):
            rows_b.append({"subject": sub, "band": band.upper(), "sd_with": sd["with_el"][k_], "sd_without": sd["without_el"][k_]})
ra, rb = pd.DataFrame(rows_a), pd.DataFrame(rows_b)
print("Reading a, precentral region per subject (SD across days of day means, dB):")
print(ra.pivot(index="subject", columns="band", values=["sd_with", "sd_without"]).to_string())
for band in ["LFB", "HFB"]:
    a_ = ra[ra.band == band]
    b_ = rb[rb.band == band]
    print(f"{band}: reading a, SD lower with baseline in {(a_.sd_with < a_.sd_without).sum()} of {len(a_)} subjects "
          f"(median {a_.sd_with.median():.3f} against {a_.sd_without.median():.3f} dB); reading b, in "
          f"{(b_.sd_with < b_.sd_without).sum()} of {len(b_)} good electrodes ({(b_.sd_with < b_.sd_without).mean():.3f}; "
          f"median {b_.sd_with.median():.3f} against {b_.sd_without.median():.3f} dB)")
print("Day effects that remain (claim fig5-recording-day-effects, band mean): significant subject and band pairs",
      int(dayk[(dayk.measure == "mean") & dayk.significant].shape[0]))

**Result: reproduced.** The SD across days is lower with baseline subtraction for 9 of 12 subjects in LFB and 12 of 12 in HFB (reading a) and for 0.918 and 0.901 of good electrodes (reading b). Variability remains: 11 subject and band pairs have significant day effects after baseline subtraction (median SD with baseline 0.20 and 0.14 dB in the precentral region).

## Claim `discussion-day-changes-small`

> Day-to-day changes in baseline-subtracted spectral power are small (about 1-2 dB) compared
> with spectral power without baseline subtraction (about 10-50 dB).

In [ ]:
rows = []
for sub in SUBJECTS:
    g = E[E.subject == sub]
    for band in ["lfb", "hfb"]:
        col = f"roi_band:precentral:{band}_mean"
        dm = g.groupby("day")[col].mean()
        raw = g[f"roi_band_raw:precentral:{band}_mean"].mean()
        rows.append({"subject": sub, "band": band.upper(), "largest_day_change_dB": round((dm - g[col].mean()).abs().max(), 3),
                     "day_mean_range_dB": round(dm.max() - dm.min(), 3), "level_uV_dB": round(raw, 2),
                     "level_V_dB": round(raw - 120, 2)})
dc = pd.DataFrame(rows)
print(dc.to_string(index=False))
print(f"\nlargest day change {dc.largest_day_change_dB.max():.3f} dB (holds at <= 2.2: {(dc.largest_day_change_dB <= 2.2).all()}); "
      f"range of day means up to {dc.day_mean_range_dB.max():.3f} dB")
print(f"levels without baseline: microvolts {dc.level_uV_dB.min():.1f} to {dc.level_uV_dB.max():.1f} dB "
      f"(within 9-55: {dc.level_uV_dB.between(9, 55).all()}); volts {dc.level_V_dB.min():.1f} to {dc.level_V_dB.max():.1f} dB "
      f"(within 9-55: {dc.level_V_dB.between(9, 55).all()})")

**Result: reproduced.** The largest deviation of a day mean from the subject mean is 1.01 dB (the range of day means reaches 1.80 dB), and the power without baseline subtraction is 12.4 to 36.3 dB with the data in microvolts, inside 9 to 55 dB; in volts it would be -107.5 to -83.7 dB.

## Regression claims (Figure 6 and Extended Data Figure 6-2)

The method is the claims file's opening comment: per subject, good electrode and band, forward
selection by OLS adjusted R2 over the design columns, then 200 day-balanced splits (90/10, Methods)
with a Huber fit of the full model and, for the Delta R2 of each selected feature, a refit with that
feature shuffled among the training events. The fits minimise sklearn's `HuberRegressor` objective
(the code's model), many at a time (helper `huber_fit`); the first cell compares them with sklearn.
Delta R2 refits are run where the full model's R2 exceeds 0.05 in either band, the electrodes Figure
6c-d show (legend); elsewhere only R2, coefficients and selection are used. The variants the claims
name are run beside the main analysis: the code's 80/20 split, the code's band median, the code's
Delta R2 (refit scored on shuffled test events), the code's selection of electrodes by LFB R2, and
the code's inclusion of bad electrodes. The code's log(x + 1) transforms are not run as a variant.

In [ ]:
# Regression inputs per subject: the kept events (reading b, as the code's power files hold),
# the design of the ten features, the electrode band power (all electrodes; the bad ones are
# used only for the code's variant), and 200 day-balanced splits for each split ratio.
rng_fig6 = np.random.default_rng(6)
E = ev[ev.keep_b]
keep = ev.keep_b.values
Y = {}
i = 0
for s, r in zip(sessions, ecog):
    for k in range(len(r["ok"])):
        if keep[i]:
            for meas in ["mean", "median"]:
                for band in ["lfb", "hfb"]:
                    Y.setdefault((s["subject"], meas, band), []).append(r["el_band"][k][f"{band}_{meas}"])
        i += 1
Y = {key: np.array(v) for key, v in Y.items()}
design, splits = {}, {}
for sub in SUBJECTS:
    g = E[E.subject == sub]
    assert len(g) == len(Y[(sub, "mean", "lfb")])
    design[sub] = ajile.regression_design(g)
    splits[sub] = {pt: ajile.day_balanced_splits(g.day.values, rng_fig6, 200, pt) for pt in (0.9, 0.8)}
REG_FEATS = list(design[SUBJECTS[0]][1])
n_el = {sub: Y[(sub, "mean", "lfb")].shape[1] for sub in SUBJECTS}
print("Features:", REG_FEATS)
print("Per subject: events, design columns, electrodes (good), test events per split at 90/10 and 80/20:")
for sub in SUBJECTS:
    print(f"  S{sub}: {len(Y[(sub, 'mean', 'lfb')])} events, {design[sub][0].shape[1]} columns, {n_el[sub]} electrodes "
          f"({int(first_good[sub].sum())} good), {splits[sub][0.9][1].shape[1]} and {splits[sub][0.8][1].shape[1]} test events")

In [ ]:
# The batched Huber fit (helper huber_fit) against sklearn's HuberRegressor, which the code uses,
# on the first 20 splits of one electrode per subject (the electrode whose LFB power correlates
# best with reach angle), with the columns forward selection keeps.
from sklearn.linear_model import HuberRegressor
rows = []
for sub in SUBJECTS:
    X, groups = design[sub]
    Xn = X.to_numpy(float)
    y = Y[(sub, "mean", "lfb")][:, int(np.nanargmax(np.abs(np.corrcoef(X["angle"], Y[(sub, "mean", "lfb")].T)[0, 1:])))]
    cols = sorted(ajile.forward_select(Xn, y))
    Z = np.column_stack([Xn[:, cols], np.ones(len(y))])
    tr, te = splits[sub][0.9]
    W, _ = ajile.huber_fit(Z[tr[:20]], y[tr[:20]])
    for k in range(20):
        m = HuberRegressor().fit(Z[tr[k], :-1], y[tr[k]])
        rows.append({"subject": sub, "coef_diff": np.abs(np.r_[m.coef_, m.intercept_] - W[k]).max(),
                     "r2_diff": abs(m.score(Z[te[k], :-1], y[te[k]]) - ajile._r2(y[te[k]], Z[te[k]] @ W[k]))})
hc = pd.DataFrame(rows)
print(f"{len(hc)} fits: largest coefficient difference {hc.coef_diff.max():.2e}, largest withheld R2 difference "
      f"{hc.r2_diff.max():.2e} (median {hc.r2_diff.median():.1e})")

In [ ]:
# The models. Stage 1 fits the full model for every electrode and band (forward selection and
# 200 splits); the Delta R2 refits are needed only where the full model's R2 exceeds 0.05 (the
# electrodes Figure 6c-d show), so stage 2 refits those, in both bands. The same for the code's
# 80/20 split (good electrodes), and the full model for the code's median band power.
def seed_of(sub, e, band, variant):
    return int(sub) * 100000 + e * 100 + (band == "hfb") * 10 + variant

def fit(keys, meas, per_train, full_only, variant):
    jobs = []
    for sub, e, band in keys:
        X, groups = design[sub]
        tr, te = splits[sub][per_train]
        jobs.append((X, groups, Y[(sub, meas, band)][:, e], tr, te, seed_of(sub, e, band, variant), full_only))
    rows = []
    for (sub, e, band), out in zip(keys, ajile.run_models(jobs)):
        row = {"subject": sub, "electrode": e, "band": band, "good": bool(first_good[sub][e])}
        if out is None:
            rows.append(row)
            continue
        row.update({"r2": out["r2"], "selected": out["selected"]})
        row.update({f"coef:{f}": v for f, v in out["coef"].items()})
        row.update({f"coefsdm:{f}": v for f, v in out["coef_sd_of_mean"].items()})
        row.update({f"keep:{f}": v for f, v in out["keep_prob"].items()})
        if not full_only:
            row.update({f"dr2:{f}": v for f, v in out["delta_r2"].items()})
            row.update({f"dr2code:{f}": v for f, v in out["delta_r2_code"].items()})
        rows.append(row)
    return pd.DataFrame(rows)

def two_stage(meas, per_train, electrodes, variant):
    keys = [(sub, e, band) for sub in SUBJECTS for e in electrodes(sub) for band in ["lfb", "hfb"]]
    full = fit(keys, meas, per_train, True, variant)
    over = full[full.r2 > 0.05][["subject", "electrode"]].drop_duplicates()
    keys2 = [(s_, e, band) for s_, e in over.itertuples(index=False) for band in ["lfb", "hfb"]]
    refit = fit(keys2, meas, per_train, False, variant)
    out = full.merge(refit[["subject", "electrode", "band"] + [c for c in refit if c.startswith("dr2")]],
                     on=["subject", "electrode", "band"], how="left")
    assert np.allclose(full.set_index(["subject", "electrode", "band"]).loc[refit.set_index(["subject", "electrode", "band"]).index].r2,
                       refit.r2)
    return out, len(keys), len(keys2)

started = time.time()
reg, n1, n2 = two_stage("mean", 0.9, lambda sub: range(n_el[sub]), 0)
t_main = time.time() - started
reg82, m1, m2 = two_stage("mean", 0.8, lambda sub: np.nonzero(first_good[sub])[0], 1)
t_82 = time.time() - started - t_main
regmed = fit([(sub, e, band) for sub in SUBJECTS for e in np.nonzero(first_good[sub])[0] for band in ["lfb", "hfb"]],
             "median", 0.9, True, 2)
print(f"main (90/10, band mean, all electrodes): {n1} full models, {n2} with refits, {t_main / 60:.1f} min; "
      f"80/20 (good electrodes): {m1} and {m2}, {t_82 / 60:.1f} min; band median: {len(regmed)} full models; "
      f"total {(time.time() - started) / 60:.1f} min")
for name, d in [("main", reg), ("80/20", reg82), ("median", regmed)]:
    print(f"{name}: models with nothing to fit (flat power) {int(d.r2.isna().sum())}, of which good {int((d.r2.isna() & d.good).sum())}; "
          f"models with no feature selected {int((d.selected.str.len() == 0).sum())}")
G = reg[reg.good]

## Claim `fig6-r2-at-most-025`

> All regression models explain at most 25% of the variance of withheld single-event power.

In [ ]:
rows = []
for name, d in [("main (90/10, band mean, good electrodes)", G), ("code's 80/20 split", reg82), ("code's band median", regmed)]:
    for band in ["lfb", "hfb"]:
        x = d[d.band == band].dropna(subset=["r2"])
        top = x.loc[x.r2.idxmax()]
        rows.append({"variant": name, "band": band.upper(), "models": len(x), "max_R2": round(top.r2, 3),
                     "at": f"S{top.subject} e{top.electrode}", "median_R2": round(x.r2.median(), 4),
                     "share_R2_over_0.05": round((x.r2 > 0.05).mean(), 3), "min_R2": round(x.r2.min(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nPer subject (main): largest R2 and electrodes with R2 > 0.05, by band")
print(G.groupby(["subject", "band"]).r2.agg(["max", lambda v: int((v > 0.05).sum()), "count"])
      .rename(columns={"<lambda_0>": "over_0.05"}).unstack().round(3).to_string())
mx = G.r2.max()
print(f"\nLargest full-model withheld R2 over good electrodes, subjects and bands: {mx:.3f}; at most 0.275: {mx <= 0.275}")
print(f"With bad electrodes too: {reg.r2.max():.3f}")
# For reference only: the R2 the files store per electrode (the paper's own results), against these.
stored = pd.concat([s["electrodes"].assign(subject=s["subject"], electrode=np.arange(len(s["electrodes"])))
                    for s in sessions if s["day"] == min(x["day"] for x in sessions if x["subject"] == s["subject"])])
for band, col in [("lfb", "low_freq_R2"), ("hfb", "high_freq_R2")]:
    m = G[G.band == band].merge(stored[["subject", "electrode", col]], on=["subject", "electrode"]).dropna(subset=["r2", col])
    print(f"stored {col}: max {stored[col].max():.3f}; Spearman with the R2 here over {len(m)} good electrodes "
          f"{stats.spearmanr(m.r2, m[col])[0]:.3f}; median difference (here minus stored) {np.median(m.r2 - m[col]):+.4f}")

**Result: reproduced.** The largest split-averaged full-model withheld R2 over the 1096 good electrodes of the 12 subjects and both bands is 0.189 (HFB, S04), at most 0.275; the largest in LFB is 0.149 (S01). With the code's 80/20 split it is 0.192, with the code's band median 0.203, and with bad electrodes included 0.189. The median R2 is -0.014 in both bands, and 2.0% (LFB) and 1.3% (HFB) of good electrodes exceed 0.05. For reference only, the R2 stored in the files reach 0.149 and 0.255, and their Spearman correlation with the values here is 0.718 and 0.643.

## Claim `fig6b-best-models-sensorimotor`

> Regression models with the highest withheld R2 are mainly at electrodes over frontoparietal
> sensorimotor areas, for both bands.

In [ ]:
FP5 = ["middle frontal", "precentral", "postcentral", "inferior parietal", "supramarginal"]
SM3 = ["precentral", "postcentral", "inferior parietal"]
TEMP = ["superior temporal", "middle temporal", "inferior temporal"]
raw_w = {sub: ajile.roi_weights(roi_mats[sub], first_good[sub], normalise=False) for sub in SUBJECTS}
def region_of(sub, e):
    w = raw_w[sub][e]
    return ajile.ROI_NAMES[int(np.argmax(w))] if w.sum() > 0 else "none"
for d in (G, reg, reg82):
    d["region"] = [region_of(s_, e) for s_, e in zip(d.subject, d.electrode)]

def project(d, col):
    """Projection-weighted mean over each subject's good electrodes, then the mean over subjects."""
    per = []
    for sub in SUBJECTS:
        v = np.zeros(n_el[sub])
        x = d[d.subject == sub]
        v[x.electrode.values] = np.nan_to_num(x[col].values.astype(float))
        with np.errstate(invalid="ignore"):
            per.append(weights[sub].T @ v)
    return pd.Series(np.nanmean(per, 0), index=ajile.ROI_NAMES), pd.Series(np.sum(np.isfinite(per), 0), index=ajile.ROI_NAMES)

holds6b = {}
for band in ["lfb", "hfb"]:
    d = G[G.band == band]
    hi = d[d.r2 > 0.05]
    proj, n_sub = project(d, "r2")
    tab = pd.DataFrame({"electrodes": d.groupby("region").size(), "R2_over_0.05": hi.groupby("region").size(),
                        "median_R2": d.groupby("region").r2.median().round(4), "projected_R2": proj.round(4),
                        "subjects_covering": n_sub}).fillna(0)
    print(f"{band.upper()}:\n{tab.to_string()}")
    share = hi.region.isin(FP5).mean()
    med_sm, med_t = d[d.region.isin(SM3)].r2.median(), d[d.region.isin(TEMP)].r2.median()
    top3 = list(proj.sort_values(ascending=False).index[:3])
    a = bool(share > 0.5 and med_sm > med_t)
    b = all(r in FP5 for r in top3)
    holds6b[band] = (a, b)
    print(f"  reading a: {len(hi)} electrodes with R2 > 0.05, share in the five frontoparietal regions {share:.3f}; "
          f"median R2 sensorimotor {med_sm:.4f} against temporal {med_t:.4f}; holds {a}")
    print(f"  reading b: three highest projected R2 {top3}; all frontoparietal: {b}")
print("Holds (reading a, reading b):", holds6b)

**Result: reproduced.** Under reading a, all 22 good electrodes with LFB R2 > 0.05 and 13 of 14 (0.929) with HFB R2 > 0.05 are assigned to the five frontoparietal regions, and the median R2 of precentral, postcentral and inferior parietal electrodes is above that of temporal electrodes in both bands (-0.0097 against -0.0158 in LFB, -0.0117 against -0.0147 in HFB). Under reading b, the three regions with the highest projected R2 are precentral, postcentral and inferior parietal in both bands. Few electrodes exceed 0.05, from 4 subjects in LFB and 6 in HFB.

## Claim `fig6c-angle-and-day-most-informative`

> Reach angle and day of recording are the most informative features for single-event power,
> especially for low-frequency power.

In [ ]:
FEAT_ORDER = ["angle", "day", "duration", "magnitude", "speed", "speech", "bimanual_ratio", "bimanual_overlap",
              "bimanual_class", "time_of_day"]
def ranked(d, band, prefix, select_band=None, fn=np.median):
    """Median over electrodes with R2 > 0.05 (in `select_band`, default the same band) of a per-feature column."""
    sb = select_band or band
    ok = d[(d.band == sb) & (d.r2 > 0.05)][["subject", "electrode"]]
    x = d[d.band == band].merge(ok, on=["subject", "electrode"])
    vals = pd.Series({f: fn(x[f"{prefix}{f}"].values) for f in FEAT_ORDER})
    return vals.sort_values(ascending=False), len(x)
variants6c = [("main", G, "dr2:", None), ("code's Delta R2", G, "dr2code:", None),
              ("code's selection by LFB R2", G, "dr2:", "lfb"), ("code's 80/20 split", reg82[reg82.good], "dr2:", None),
              ("with bad electrodes", reg, "dr2:", None)]
tabs, top2 = {}, {}
for name, d, prefix, sb in variants6c:
    for band in ["lfb", "hfb"]:
        v, n = ranked(d, band, prefix, sb)
        tabs[(name, band.upper(), n)] = v.round(4)
        top2[(name, band)] = list(v.dropna().index[:2])
print("Median Delta R2 over electrodes with R2 > 0.05 (variant, band, electrodes):")
print(pd.DataFrame(tabs).loc[FEAT_ORDER].to_string())
for k, v in top2.items():
    print(f"  {k[0]}, {k[1].upper()}: two largest {v}; angle and day both: {set(v) == {'angle', 'day'}}")

**Result: reproduced.** Over the 22 good electrodes with LFB R2 > 0.05, reach angle (median Delta R2 0.066) and recording day (0.017) have the two largest median Delta R2; time of day follows (0.0017) and the seven other features have a median of 0. In HFB (14 electrodes) day (0.068) and angle (0.033) are the two largest. Every variant (the code's Delta R2, selection by LFB R2, the 80/20 split, bad electrodes included) gives angle and day as the two largest in both bands.

## Claim `extfig6-2-angle-most-retained`

> Reach angle is the feature most often retained by forward selection in sensorimotor regions,
> followed by reach duration, reach magnitude, bimanual ratio and bimanual overlap.

In [ ]:
keep_proj = {}
for band in ["lfb", "hfb"]:
    d = G[G.band == band]
    keep_proj[band] = pd.DataFrame({f: project(d, f"keep:{f}")[0] for f in FEAT_ORDER})
    print(f"{band.upper()}: retention probability projected onto the regions (mean over subjects)")
    print(keep_proj[band].round(3).to_string())
angle_top = {}
for band in ["lfb", "hfb"]:
    for r in SM3:
        row = keep_proj[band].loc[r]
        angle_top[(band, r)] = bool(row["angle"] > row.drop("angle").max())
print("\nAngle above every other feature:", angle_top, "->", all(angle_top.values()))
ORDER = ["duration", "magnitude", "bimanual_ratio", "bimanual_overlap"]
order_holds = {}
for lab, regs in [("a, three sensorimotor regions", SM3), ("b, all eight regions", ajile.ROI_NAMES)]:
    m = pd.concat([keep_proj[b].loc[regs] for b in ["lfb", "hfb"]]).mean().sort_values(ascending=False)
    order_holds[lab[0]] = list(m.index[1:5]) == ORDER
    print(f"reading {lab}: ranking {[(f, round(v, 3)) for f, v in m.items()]}; second to fifth {list(m.index[1:5])}; "
          f"order holds {order_holds[lab[0]]}")
print("Retention over good electrodes, not projected (mean over electrodes, both bands):",
      G[[f"keep:{f}" for f in FEAT_ORDER]].mean().round(3).to_dict())

**Result: partially reproduced.** Reach angle has the highest projected retention probability in precentral, postcentral and inferior parietal in both bands (0.448 to 0.679, against at most 0.400 for any other feature in these regions). The order of the next four does not hold: averaged over the three sensorimotor regions and both bands, the second to fifth features are onset speed, speech ratio, reach magnitude and bimanual ratio, and over all eight regions speech ratio, onset speed, reach magnitude and bimanual ratio; reach duration is sixth and seventh. The second to tenth features lie close together (0.202 to 0.357).

## Claim `fig6d-upward-reaches-enhance-pattern`

> Upward reaches are associated with lower low-frequency power and higher high-frequency power,
> that is, a larger version of the average movement response.

In [ ]:
rows = []
for name, d, sb in [("main", G, None), ("code's selection by LFB R2", G, "lfb"), ("with bad electrodes", reg, None),
                    ("code's 80/20 split", reg82[reg82.good], None)]:
    for band in ["lfb", "hfb"]:
        ok = d[(d.band == (sb or band)) & (d.r2 > 0.05)][["subject", "electrode"]]
        x = d[d.band == band].merge(ok, on=["subject", "electrode"])["coef:angle"]
        rows.append({"variant": name, "band": band.upper(), "electrodes": len(x), "median_angle_coef": round(x.median(), 4),
                     "share_negative": round((x < 0).mean(), 3), "q25": round(x.quantile(.25), 4), "q75": round(x.quantile(.75), 4)})
a6d = pd.DataFrame(rows)
print(a6d.to_string(index=False))
m = a6d[a6d.variant == "main"].set_index("band").median_angle_coef
print(f"main: LFB median below 0 {m['LFB'] < 0}; HFB median above 0 {m['HFB'] > 0}")
print("Median angle coefficient by subject (main, electrodes with R2 > 0.05 in the band):")
print(G[G.r2 > 0.05].groupby(["subject", "band"])["coef:angle"].median().unstack().round(3).to_string())

**Result: reproduced.** Over good electrodes with R2 > 0.05 in the band, the median reach-angle coefficient is -1.027 in LFB (22 electrodes, 95.5% negative) and 0.535 in HFB (14 electrodes, none negative). The code's selection by LFB R2 gives -1.027 and 0.202, bad electrodes included -0.815 and 0.515, and the 80/20 split -0.973 and 0.540, so the signs hold in every variant. By subject the LFB medians are -1.353 to 0 (4 subjects) and the HFB medians 0 to 0.730 (6 subjects).

## Claim `fig6d-angle-and-day-largest-coefficients`

> Reach angle and day of recording have the largest coefficient magnitudes among the features,
> and the SD of the recording-day coefficients is large.

In [ ]:
res6d = {}
for name, d, sb in [("main", G, None), ("code's selection by LFB R2", G, "lfb"), ("with bad electrodes", reg, None),
                    ("code's 80/20 split", reg82[reg82.good], None)]:
    tab = {}
    for band in ["lfb", "hfb"]:
        va, n = ranked(d, band, "coef:", sb, fn=lambda v: np.median(np.abs(v)))
        vb, _ = ranked(d, band, "coef:", sb, fn=lambda v: abs(np.median(v)))
        tab[(band.upper(), "a: median |coef|", n)] = va.round(4)
        tab[(band.upper(), "b: |median coef|", n)] = vb.round(4)
        res6d[(name, band, "a")] = list(va.dropna().index[:2])
        res6d[(name, band, "b")] = list(vb.dropna().index[:2])
    print(f"{name}:\n{pd.DataFrame(tab).loc[FEAT_ORDER].to_string()}")
for k, v in res6d.items():
    print(f"  {k[0]}, {k[1].upper()}, reading {k[2]}: two largest {v}; angle and day: {set(v) == {'angle', 'day'}}")

**Result: reproduced.** Over good electrodes with R2 > 0.05, reach angle and the SD of the recording-day coefficients have the two largest values under both readings in both bands: in LFB angle 1.027 and day 0.660 (then time of day 0.188), in HFB day 0.733 and angle 0.535 (then reach magnitude 0.231 under reading a). The code's selection by LFB R2, the inclusion of bad electrodes and the 80/20 split give the same two in both bands and readings. The day SD is the second largest value in LFB and the largest in HFB.

In [ ]:
# The fig6b-d criteria on the electrodes the stored R2 select (the paper's own per-electrode R2 in
# the files, low_freq_R2 and high_freq_R2, above 0.05), and the overlap of those sets with the
# sets selected by the R2 here. Delta R2 refits are run for the stored-selected good electrodes
# that lack them, with the seeds of the main fit.
st = pd.concat([stored[["subject", "electrode", col]].rename(columns={col: "r2_stored"}).assign(band=band)
                for band, col in [("lfb", "low_freq_R2"), ("hfb", "high_freq_R2")]])
Gs = G.merge(st, on=["subject", "electrode", "band"], how="left")
for band in ["lfb", "hfb"]:
    d = Gs[Gs.band == band]
    ours = set(map(tuple, d[d.r2 > 0.05][["subject", "electrode"]].values))
    theirs = set(map(tuple, d[d.r2_stored > 0.05][["subject", "electrode"]].values))
    print(f"{band.upper()}: R2 > 0.05 here {len(ours)} electrodes ({len({s_ for s_, _ in ours})} subjects), stored {len(theirs)} "
          f"({len({s_ for s_, _ in theirs})} subjects), in both {len(ours & theirs)}; by subject here "
          f"{dict(pd.Series([s_ for s_, _ in ours]).value_counts().sort_index())}, stored "
          f"{dict(pd.Series([s_ for s_, _ in theirs]).value_counts().sort_index())}")
need = Gs[(Gs.r2_stored > 0.05) & Gs["dr2:angle"].isna()][["subject", "electrode"]].drop_duplicates()
extra = fit([(s_, e, band) for s_, e in need.itertuples(index=False) for band in ["lfb", "hfb"]], "mean", 0.9, False, 0)
print(f"refits for stored-selected electrodes without them: {len(need)} electrodes")
if len(extra):
    ix = ["subject", "electrode", "band"]
    chk = Gs.set_index(ix).loc[extra.set_index(ix).index]
    assert np.allclose(chk.r2, extra.r2)
    Gs = Gs.set_index(ix)
    dcols = [c for c in extra if c.startswith("dr2")]
    Gs.loc[extra.set_index(ix).index, dcols] = extra.set_index(ix)[dcols].values
    Gs = Gs.reset_index()
# Select by the stored R2: ranked() and the angle medians select on the column r2.
S_ = Gs.assign(r2_here=Gs.r2, r2=Gs.r2_stored)
print("\nfig6b reading a with the stored selection (share of selected electrodes in the five frontoparietal regions):")
for band in ["lfb", "hfb"]:
    hi = S_[(S_.band == band) & (S_.r2 > 0.05)]
    print(f"  {band.upper()}: {hi.region.isin(FP5).sum()} of {len(hi)} ({hi.region.isin(FP5).mean():.3f}); regions "
          f"{dict(hi.region.value_counts())}")
print("For reference only (the paper's own values): median stored R2 sensorimotor against temporal, and the three "
      "highest projected stored R2:")
for band in ["lfb", "hfb"]:
    d = S_[S_.band == band]
    proj, _ = project(d, "r2")
    print(f"  {band.upper()}: {d[d.region.isin(SM3)].r2.median():.4f} against {d[d.region.isin(TEMP)].r2.median():.4f}; "
          f"{list(proj.sort_values(ascending=False).index[:3])}")
print("\nfig6c with the stored selection, median Delta R2:")
for band in ["lfb", "hfb"]:
    for prefix in ["dr2:", "dr2code:"]:
        v, n = ranked(S_, band, prefix)
        print(f"  {band.upper()} {prefix} ({n} electrodes): {v.round(4).head(4).to_dict()}; angle and day the two largest: "
              f"{set(v.index[:2]) == {'angle', 'day'}}")
print("\nfig6d with the stored selection:")
for band in ["lfb", "hfb"]:
    x = S_[(S_.band == band) & (S_.r2 > 0.05)]["coef:angle"]
    va, n = ranked(S_, band, "coef:", fn=lambda v: np.median(np.abs(v)))
    vb, _ = ranked(S_, band, "coef:", fn=lambda v: abs(np.median(v)))
    print(f"  {band.upper()} ({n} electrodes): median angle coefficient {x.median():+.4f} ({(x < 0).mean():.3f} negative); "
          f"reading a {va.round(4).head(3).to_dict()}; reading b {vb.round(4).head(3).to_dict()}; "
          f"angle and day the two largest: a {set(va.index[:2]) == {'angle', 'day'}}, b {set(vb.index[:2]) == {'angle', 'day'}}")
# The day and time-of-day coefficient: the mean over splits of the SD of the one-hot coefficients
# (used above) against the SD of the split-averaged one-hot coefficients, on both selections.
print("\nDay and time-of-day value under the two definitions (reading a, median |value|):")
for lab, d in [("selection here", G), ("stored selection", S_)]:
    for band in ["lfb", "hfb"]:
        va, n = ranked(d, band, "coef:", fn=lambda v: np.median(np.abs(v)))
        vs, _ = ranked(d, band, "coefsdm:", fn=lambda v: np.median(np.abs(v)))
        print(f"  {lab}, {band.upper()} ({n}): mean of per-split SD day {va['day']:.4f}, time of day {va['time_of_day']:.4f}; "
              f"SD of mean day {vs['day']:.4f}, time of day {vs['time_of_day']:.4f}; two largest with the SD of mean "
              f"{list(vs.index[:2])}")


**Electrodes selected by the stored R2 (sensitivity, not the criterion).** The electrodes with R2 > 0.05 here and those with stored R2 > 0.05 overlap in 18 electrodes in LFB (22 here, 23 stored) and 12 in HFB (14 here, 22 stored). The stored sets come from 5 subjects in LFB and 8 in HFB, against 4 and 6 here. With the stored selection, 22 of 23 LFB and 18 of 22 HFB electrodes are in the five frontoparietal regions (fig6b, reading a, share part). Angle (0.063) and day (0.013) have the two largest median Delta R2 in LFB, and day (0.050) and angle (0.019) in HFB, with the code's Delta R2 too (fig6c). The median angle coefficient is -1.026 in LFB and +0.498 in HFB (fig6d, upward reaches). Angle and the day SD are the two largest coefficient values under both readings in both bands (fig6d, largest coefficients). The fig6b-d results are the same on either set of electrodes. Taken as the SD of the split-averaged one-hot coefficients, not the mean over splits of the per-split SD, the day value is 0.658 against 0.660 in LFB and 0.729 against 0.733 in HFB, and angle and day stay the two largest.


In [ ]:
# How much the assumed reading of the bad-segment rule matters: the main ECoG numbers with only
# the segments with missing samples dropped (no 0 dB or 115-125 Hz rule).
E_alt = ev[ev.ok & ~ev.bad_missing]
print("events:", len(E_alt), "against", len(E))
for band in ["lfb", "hfb"]:
    col = f"roi_band:precentral:{band}_mean"
    h, p = stats.kruskal(*[g[col].values for _, g in E_alt.groupby("subject")])
    ins = sum(TABLE1[s][0 if band == "lfb" else 1][0] <= E_alt[E_alt.subject == s][col].mean() <= TABLE1[s][0 if band == "lfb" else 1][1]
              for s in SUBJECTS)
    sig = [s for s in SUBJECTS if stats.kruskal(*[d[col].values for _, d in E_alt[E_alt.subject == s].groupby("day")])[1] < 0.05]
    print(f"{band.upper()}: across subjects p = {p:.3g}; Table 1 means inside {ins} of 12; day effects in {sig}")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| subject-days (link) | reproduced |
| roi-projection-link (link, Figshare projection matrices) | reproduced |
| cohort | partially reproduced (seven subjects with depth electrodes, not five; sex and age not in the dataset) |
| fig2-event-counts | not reproduced (211 to 996 events per subject, median 770.5) |
| fig2-features-differ-across-subjects | reproduced |
| fig2-reach-shape | partially reproduced (angle peaks near +-70, not +-90) |
| fig2-bimanual-skew | reproduced |
| extfig2-1c-contralateral-larger | reproduced |
| extfig2-2-feature-correlations | partially reproduced (5 of 6) |
| fig2-days-and-waking | partially reproduced (S04's busiest day holds 38%) |
| fig3-group-lfb-decrease-hfb-increase | reproduced |
| fig3-sensorimotor-largest | reproduced |
| fig3-lfb-more-widespread | blocked, methods underspecified (readings disagree) |
| fig4-precentral-subject-variability | reproduced (one tie broken by interpolated width) |
| fig5-band-power-differs-across-subjects | reproduced |
| table1-subject-band-power-ci | partially reproduced (23 of 24) |
| fig5-recording-day-effects | reproduced (24 of 24) |
| extfig5-1-baseline-reduces-day-variability | reproduced |
| discussion-day-changes-small | reproduced |
| fig6-r2-at-most-025 | reproduced (largest R2 0.189) |
| fig6b-best-models-sensorimotor | reproduced |
| fig6c-angle-and-day-most-informative | reproduced |
| extfig6-2-angle-most-retained | partially reproduced (angle first; the order of the next four differs) |
| fig6d-upward-reaches-enhance-pattern | reproduced |
| fig6d-angle-and-day-largest-coefficients | reproduced |

**How the analysis differs from the paper's.** The paper's preprocessing, event pruning and visual
removal of false positives were done before the data were deposited; the dataset holds 8088 events and
marks none as false positives, so all are kept, for the behavioural claims and for every ECoG claim. The bad-segment rules use an assumed summary of
segment power; with only segments with missing samples dropped, the across-subject tests, the day
effects and the Table 1 outcome (22 of 24 inside) do not change (last data check above). Region power uses the paper's Figshare projection matrices, not a new
projection from the AAL atlas. Bad electrodes are those of each subject's first file. The Morlet
transform is computed by FFT and matches MNE's direct convolution. The regression models minimise the objective of sklearn's HuberRegressor by a batched Newton method, which matches sklearn's fits to 4.4e-05 in withheld R2; Delta R2 refits are run only at electrodes whose full-model R2 exceeds 0.05, the electrodes Figure 6c-d show. The day and time-of-day coefficient is the mean over splits of the SD of their one-hot coefficients; the SD of the split-averaged coefficients changes it by at most 0.004. On the electrodes selected by the R2 stored in the files, the fig6b-d results are the same. The code's log(x + 1) transforms of reach magnitude and onset speed are not run as a variant. In the code's Delta R2 variant the refit is shared with the main version and only the test events are shuffled.

**What was found about the dataset.** (1) In three files (S01 day 7, S04 day 3, S06 day 3) the ECoG
covers only the end of the day but its `starting_time` is 0, while the event table and pose use a
clock from midnight; the ECoG must be shifted by 8.1 to 8.5 h, as `ReachEvents` and the neural
response show. (2) The files hold 8088 events; the data descriptor states 6931 verified events, and
the paper's counts per subject are lower than the files'. (3) The reach angle is stored in image
coordinates (upward negative) and unfolded, without saying so. (4) The pose is labelled meters but
is in pixels. (5) `session_start_time` gives midnight UTC labelled as 19:00 at UTC-5. (6) Bimanual
overlap is described as a fraction but reaches 3.83 (it is in seconds). (7) 124 event segments contain
missing ECoG samples. (8) Sex and age are not recorded. (9) Seven subjects have electrode groups
described as depth, where the paper says five.